# Vélib' station availability: analysis notebook

Group H (ESSEC, Data, Data storage). **Question: how often does a Vélib' station fail its users (no bike to rent, or no free dock to return) while no other station within 300 m can serve them, and does this differ between Paris and the surrounding communes?**

This notebook is the full analysis of the project. It runs on VM1 (main series) or on VM2 (replication), set by `VM` in the first code cell. It follows the story of the report, one section per report section, and ends with a replay of the whole collection period. Part 0 is the workflow (get the data, check it, build the small tables), run once. Sections 1 to 10 tell the story and each ends with one figure or table. Read `notebooks/README.md` first (set-up for macOS and Windows).

| Section | Report / slides | Output (`outputs/`) | Grading dimension |
|---|---|---|---|
| **Part 0** | workflow: data, self-check, small tables (run once) | `data/derived/` | reproducibility |
| 1. Question and amendments | introduction: why, what changed after each feedback | definitions, "what we changed" table | question, reflection on feedback |
| 2. Data and collection | data | `fig_coverage.png` | collection |
| 3. Quality and cleaning | method | `fig_cleaning.png` | cleaning |
| 4. Storage | method | `storage_benchmark.csv` | storage |
| 5. Sampling | method, limits | `fig_sampling.png` | collection, analysis |
| 6. **Validation with events** | results (first) | `fig_events_timeline.png` | analysis, communication |
| 7. Analysis: how often, when, where, 300 m, robustness | results | `fig_hour_weekday.png`, `fig_station_map.png`, tables | analysis |
| 8. Reproducibility | method | | reproducibility |
| 9. Conclusion, limits, lessons | conclusion | | communication |
| 10. The replay | closing slide | `replay_velib.mp4`, snapshots | communication |

**Rules of the project (do not reopen)**
- VM1 is the main series. VM2 is a reliability check. **Never merge them.** All times are UTC unless a name says Paris.
- **Flag, do not alter.** `in_panel` is the single selector for the analysis rows.
- Say *cannot rent* and *cannot take a return*, never "broken".
- Population is context only, never "people affected".
- Observations and interpretations are kept apart: tables and figures state what the data show, the text under them says what we think it means.
- Never load the 25 M-row table into pandas. Query it with DuckDB (`load_status`), aggregate once, save the small result (`cached`).

## Setup

Settings are in the first block. While you develop, set `SAMPLE_DAYS` to two or three dates: every heavy step then runs in seconds on those days only (derived files get a `__sample` suffix, so they never mix with the full-period ones). Set it back to `None` for the numbers that go in the report.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, sqlite3, statistics, sys, tarfile, time, warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

for _w in (FutureWarning, DeprecationWarning, getattr(pd.errors, "Pandas4Warning", FutureWarning)):
    warnings.filterwarnings("ignore", category=_w)   # pandas deprecation notices (red boxes) do not change any result

# ---- settings you may change ---------------------------------------------------------
SAMPLE_DAYS = None      # example: ["2026-10-05", "2026-10-06"] while developing; None = full period
FORCE = False              # True = rebuild derived files even if they already exist
STOP_ON_MISMATCH = True    # stop if the self-check (0.2) does not reproduce the known numbers
MEMORY_LIMIT = "2GB"       # DuckDB memory cap; lower it on a laptop with 4-8 GB of RAM
THREADS = 2
RUN_BENCHMARK = False   # storage benchmark (section 4) is slow and needs disk: switch on on purpose
RUN_REPLAY = False        # replay video (section 10) takes minutes: switch on on purpose
# --------------------------------------------------------------------------------------

VM = "vm1"                 # main series. vm2 is a reliability check, never merged
ZENODO_RECORD = "24444738"                      # version DOI 10.5281/zenodo.24444738
ZENODO_FILES = f"https://zenodo.org/records/{ZENODO_RECORD}/files"

# repository root = the folder that contains notebooks/ (works if Jupyter was started in either)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
DERIVED_ROOT = DATA / "derived"
VM_DIR, EXT, DERIVED, DL = DATA / VM, DATA / "external", (DERIVED_ROOT if VM == "vm1" else DERIVED_ROOT / VM), DATA / "zenodo"
OUT = ROOT / "outputs"
for p in (VM_DIR, EXT, DERIVED, DL, OUT, DATA / "tmp"):
    p.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()                                   # in memory; nothing is written to disk
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{(DATA / 'tmp').as_posix()}'")   # lets DuckDB spill to disk
con.execute("SET TimeZone='UTC'")                        # every timestamp read as UTC

print("repository root:", ROOT)
# Windows refuses file paths over 260 characters, and pip's Jupyter files are long: a deep folder breaks the install
if len(str(ROOT)) > 90 or " " in str(ROOT) or "onedrive" in str(ROOT).lower():
    print("WARNING: this folder path is long, has spaces or sits in OneDrive. On Windows this can break installs and file access.\n"
          "         Prefer a short folder such as C:\\Users\\<you>\\velib (no spaces, outside OneDrive).")
print("duckdb", duckdb.__version__, "| pandas", pd.__version__, "| python", platform.python_version())
print("SAMPLE_DAYS:", SAMPLE_DAYS or "None (full period)")

In [ ]:
# ---- helpers: download, unpack, find files ------------------------------------------
def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()

def download(url, dest):
    # stream to dest.part, then rename: an interrupted download never looks complete
    dest = Path(dest)
    part = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, done, last = int(r.headers.get("content-length", 0)), 0, time.time()
        with open(part, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
                done += len(chunk)
                if time.time() - last > 5:
                    print(f"  {dest.name}: {done/1e6:,.0f} / {total/1e6:,.0f} MB", end="\r")
                    last = time.time()
    part.replace(dest)
    print(f"  {dest.name}: {dest.stat().st_size/1e6:,.0f} MB downloaded" + " " * 20)

def read_sums(path):
    # SHA256SUMS.txt: one "hash  filename" per line
    sums = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 2:
            sums[Path(parts[-1].lstrip("*")).name] = parts[0].lower()
    return sums

def fetch_checked(name, sums):
    dest = DL / name
    if dest.exists() and sums.get(name) and sha256(dest) == sums[name]:
        print(f"  {name}: already downloaded, checksum OK")
        return dest
    download(f"{ZENODO_FILES}/{name}?download=1", dest)
    if name in sums:
        if sha256(dest) != sums[name]:
            dest.unlink()
            raise RuntimeError(f"Checksum mismatch for {name}: download deleted, run the cell again.")
        print(f"  {name}: checksum OK")
    else:
        print(f"  WARNING: {name} not listed in SHA256SUMS.txt, checksum not verified")
    return dest

def unpack(archive, dest, skip=("daily_json/",)):
    # extract member by member; skip the big daily CSV copies we do not need
    mode = "r:gz" if str(archive).endswith(".gz") else "r:"
    dest = Path(dest).resolve()
    n = 0
    with tarfile.open(archive, mode) as tf:
        for m in tf:
            if any(s.strip("/") in Path(m.name).parts for s in skip):
                continue
            target = (dest / m.name).resolve()
            if dest not in target.parents and target != dest:
                raise RuntimeError(f"Unsafe path in archive: {m.name}")
            try:
                tf.extract(m, dest, filter="data")
            except TypeError:                    # older Python without the filter argument
                tf.extract(m, dest)
            n += 1
    print(f"  unpacked {n} entries from {Path(archive).name}")

def locate(name, root=VM_DIR, required=True):
    # find a file or folder by name anywhere under data/vm1 (robust to the archive layout)
    hits = sorted(root.rglob(name), key=lambda p: len(p.parts))
    if hits:
        return hits[0]
    if required:
        raise FileNotFoundError(f"'{name}' not found under {root}. Run section 0.1 (get the data).")
    return None

## 0.1 Get the data

Data are on Zenodo, version DOI 10.5281/zenodo.24444738 (licence ODbL). This cell downloads the two archives we need (about 750 MB), checks them against `SHA256SUMS.txt`, and unpacks them into `data/vm1/`. It does nothing if the data are already there (for instance if you rebuilt them with `code/run_all.py`). The 2 GB of daily CSV copies inside `vm1_csv_logs.tar.gz` are skipped.

In [ ]:
ESSENTIAL = ["station_status_clean", "episodes.parquet", "station_neighbours.csv", "threshold_sensitivity.csv"]

def data_present():
    return all(locate(n, required=False) for n in ESSENTIAL)

if data_present():
    print(f"data/{VM} already holds the data: nothing to download.")
else:
    print("Downloading from Zenodo (about 750 MB, once)...")
    download(f"{ZENODO_FILES}/SHA256SUMS.txt?download=1", DL / "SHA256SUMS.txt")
    sums = read_sums(DL / "SHA256SUMS.txt")
    for name in [f"{VM}_parquet.tar", f"{VM}_csv_logs.tar.gz"]:
        archive = fetch_checked(name, sums)
        unpack(archive, VM_DIR)
    if not data_present():
        raise RuntimeError("Unpacked, but expected files are missing: " + ", ".join(
            n for n in ESSENTIAL if not locate(n, required=False)))
    print("Done. You can delete data/zenodo/ to free about 750 MB.")

CLEAN_DIR = locate("station_status_clean")
EPISODES = locate("episodes.parquet")
NEIGH = locate("station_neighbours.csv")
THRESH = locate("threshold_sensitivity.csv")
CLEAN_GLOB = CLEAN_DIR.as_posix() + "/date=*/part-0.parquet"
n_days = len(list(CLEAN_DIR.glob("date=*")))
print(f"cleaned table: {CLEAN_DIR.relative_to(ROOT)} ({n_days} daily files)")

## Helpers for the large table

`load_status` is the only way row-level data should be read: it returns a DuckDB relation over the daily Parquet files with **only the columns you ask for** and, by default, only `in_panel` rows. Nothing is loaded into memory until you aggregate. `cached` runs a query once, saves the small result in `data/derived/`, and reloads it next time.

In [ ]:
# columns of the cleaned table (DATA_DICTIONARY.md, section 3), plus `date` from the folder name
STATUS_COLUMNS = {
    "fetched_at", "station_id", "stationCode", "last_reported", "lr_age_min", "vm", "name", "lat", "lon",
    "capacity", "num_bikes_available", "mechanical", "ebike", "num_docks_available", "docks_unavailable",
    "is_installed", "is_renting", "is_returning", "state", "out_of_service", "can_rent", "can_return",
    "partial_capture", "in_station_list", "temporary", "event_station", "capacity_conflict", "silent",
    "frozen", "in_panel", "date",
}

def day_files(days=None):
    if days is None:
        return [CLEAN_GLOB]
    files = [(CLEAN_DIR / f"date={d}" / "part-0.parquet").as_posix() for d in days]
    missing = [f for f in files if not Path(f).exists()]
    if missing:
        raise FileNotFoundError("No data for: " + ", ".join(Path(f).parent.name for f in missing))
    return files

def sql_list(items):
    return "[" + ", ".join("'" + str(i).replace("'", "''") + "'" for i in items) + "]"

def load_status(columns, panel_only=True, days="default", name=None):
    # DuckDB relation over the cleaned table. Only the listed columns. Respects SAMPLE_DAYS.
    if days == "default":
        days = SAMPLE_DAYS
    bad = [c for c in columns if c not in STATUS_COLUMNS]
    if bad:
        raise ValueError(f"Unknown column(s) {bad}. See DATA_DICTIONARY.md section 3.")
    cols = ", ".join(dict.fromkeys(columns))
    where = " WHERE in_panel" if panel_only else ""
    rel = con.sql(f"SELECT {cols} FROM read_parquet({sql_list(day_files(days))}, hive_partitioning = true){where}")
    if name:
        rel.create_view(name, replace=True)
    return rel

def cached(name, build, sample_dependent=True):
    # run `build` once (a SQL string for DuckDB, or a function returning a DataFrame), save as parquet
    suffix = "__sample" if (SAMPLE_DAYS and sample_dependent) else ""
    path = DERIVED / f"{name}{suffix}.parquet"
    if path.exists() and not FORCE:
        print(f"[cached] {path.name} ({path.stat().st_size/1e6:.1f} MB). FORCE = True to rebuild.")
        return pd.read_parquet(path)
    t0 = time.time()
    df = con.execute(build).df() if isinstance(build, str) else build()
    df.to_parquet(path, index=False)
    print(f"[built]  {path.name}: {len(df):,} rows, {path.stat().st_size/1e6:.1f} MB, {time.time()-t0:.0f} s")
    return df

## 0.2 Self-check

Reproduces the numbers published in `docs/findings_so_far.md` from the files you just unpacked. **If one differs, stop and find out why before analysing anything.** The check always runs on the full period, whatever `SAMPLE_DAYS` says. Takes a few seconds to a minute.

In [ ]:
q = con.execute(f"""
    SELECT count(*)                                              AS rows,
           count(DISTINCT fetched_at)                            AS captures,
           count(*) FILTER (WHERE in_panel)                      AS panel_rows,
           count(DISTINCT station_id) FILTER (WHERE in_panel)    AS panel_stations,
           100 * avg((NOT can_rent)::INT)   FILTER (WHERE in_panel) AS pct_cannot_rent,
           100 * avg((NOT can_return)::INT) FILTER (WHERE in_panel) AS pct_cannot_return
    FROM read_parquet('{CLEAN_GLOB}', hive_partitioning = true)
""").df().iloc[0]

ep = con.execute(f"SELECT side, count(*) AS n FROM read_parquet('{EPISODES.as_posix()}') GROUP BY side").df().set_index("side")["n"]
thr = pd.read_csv(THRESH)
share300 = float(thr.loc[thr["threshold_m"] == 300, "share_with_neighbour_%"].iloc[0])

rows = [
    ("Rows (cleaned table)",                        24_998_625, int(q["rows"]),            "n"),
    ("Status captures",                             16_452,     int(q["captures"]),        "n"),
    ("Rows in the panel (in_panel)",                24_740_954, int(q["panel_rows"]),      "n"),
    ("Stations in the panel",                       1_515,      int(q["panel_stations"]),  "n"),
    ("% of time a station cannot rent",             6.94,       round(float(q["pct_cannot_rent"]), 2),   "%"),
    ("% of time a station cannot take a return",    5.21,       round(float(q["pct_cannot_return"]), 2), "%"),
    ("Episodes no_rent",                            124_950,    int(ep.get("no_rent", 0)),   "n"),
    ("Episodes no_return",                          77_903,     int(ep.get("no_return", 0)), "n"),
    ("Episodes in total",                           202_853,    int(ep.sum()),               "n"),
    ("% of stations with a neighbour within 300 m", 68.1,       share300,                    "%"),
]
fmt = lambda v, kind: f"{v:,}" if kind == "n" else f"{v:.2f}"
checks = pd.DataFrame({
    "quantity (VM1)": [r[0] for r in rows],
    "expected": [fmt(r[1], r[3]) for r in rows],
    "found": [fmt(r[2], r[3]) for r in rows],
    "ok": [abs(r[1] - r[2]) < 1e-9 for r in rows],
})
print(checks.to_string(index=False))
if VM != "vm1":
    print("\n(VM2 run: the reference numbers above are VM1's; small differences are expected.)")
elif checks["ok"].all():
    print("\nSELF-CHECK PASSED: this setup reproduces the published numbers.")
else:
    msg = "SELF-CHECK FAILED: " + ", ".join(checks.loc[~checks["ok"], "quantity (VM1)"]) + ". Find out why before analysing."
    if STOP_ON_MISMATCH:
        raise RuntimeError(msg)
    print("\n" + msg + " (STOP_ON_MISMATCH is False, continuing)")

## 0.3 Station list

The panel stations (the 1,515 regular stations selected by `in_panel`) with their coordinates. Coordinates and names come from `station_neighbours.csv`, which was built from the last hourly station list of the period.

In [ ]:
def build_panel_stations():
    ids = con.execute(f"""
        SELECT DISTINCT station_id
        FROM read_parquet('{CLEAN_GLOB}', hive_partitioning = true) WHERE in_panel ORDER BY station_id""").df()
    nb = pd.read_csv(NEIGH)[["station_id", "stationCode", "name", "lat", "lon", "capacity"]]
    return ids.merge(nb, on="station_id", how="left")

panel_stations = cached("panel_stations", build_panel_stations, sample_dependent=False)
no_coord = int(panel_stations["lat"].isna().sum())
print(f"{len(panel_stations):,} panel stations, {no_coord} without coordinates")
assert no_coord == 0, "some panel stations are missing from station_neighbours.csv"
panel_stations.head()

## 0.4 External data: commune and population per station

The question compares Paris with the surrounding communes, so every station needs a commune. This is **not in the Zenodo record**. Plan: `external-data-plan` in the project notes and `data/external/SOURCES.md` (template, fill it in as you download).

Do the join **once** and commit the small result `data/external/station_communes.csv` (one row per panel station): `station_id, insee_code, commune, is_paris, population, population_year`. The rest of the notebook then needs neither the network nor geo libraries. The cell below checks the file if it exists; the next cell is yours to fill in.

In [ ]:
COMMUNES_CSV = EXT / "station_communes.csv"
communes = None
if COMMUNES_CSV.exists():
    communes = pd.read_csv(COMMUNES_CSV, dtype={"insee_code": str})
    need = {"station_id", "insee_code", "commune", "is_paris", "population", "population_year"}
    assert need <= set(communes.columns), f"station_communes.csv lacks: {need - set(communes.columns)}"
    assert communes["station_id"].is_unique, "more than one commune for a station"
    missing = set(panel_stations["station_id"]) - set(communes["station_id"])
    assert not missing, f"{len(missing)} panel stations have no commune"
    print(f"station_communes.csv OK: {len(communes):,} stations, {communes['commune'].nunique()} communes, "
          f"{int(communes['is_paris'].sum())} in Paris")
else:
    print("TODO: data/external/station_communes.csv does not exist yet (section 0.4). "
          "Sections that need it are skipped.")

In [ ]:
# Build the commune table once. The script code/build_station_communes.py asks API Géo (geo.api.gouv.fr)
# which commune contains each station and what its population is; it caches the answers, so it can be re-run.
# Set BUILD_COMMUNES = True, run this cell (5-10 minutes, needs the internet), then re-run the cell above.
# Afterwards: fill in population_year, complete data/external/SOURCES.md, and check by hand the stations
# on the Paris boundary (Boulogne, Neuilly, Montreuil...) and those listed in station_communes_unmatched.csv.
BUILD_COMMUNES = False
if BUILD_COMMUNES and not COMMUNES_CSV.exists():
    import subprocess, sys
    script = ROOT / "code" / "build_station_communes.py"
    subprocess.run([sys.executable, str(script)], check=True, cwd=ROOT)

## 0.5 The 300 m pair table

For the "viable alternative" rule we need, for every station, the list of other stations within 300 m. `station_neighbours.csv` only holds counts per station, so the pairs are built here, with the **same distance formula as `code/neighbours.py`** (equirectangular projection, R = 6,371,000 m, straight line). The pair table is first built over all stations of the list and checked against `n_within_300m`; it is then restricted to panel stations, because the alternative must itself be a panel station.

In [ ]:
def pairs_within(st, radius_m):
    # directed pairs (station_id -> neighbour_id) closer than radius_m; same formula as code/neighbours.py
    R = 6371000.0
    lat, lon = np.radians(st["lat"].to_numpy()), np.radians(st["lon"].to_numpy())
    x, y = R * lon * np.cos(lat.mean()), R * lat
    d = np.sqrt((x[:, None] - x[None, :]) ** 2 + (y[:, None] - y[None, :]) ** 2)
    np.fill_diagonal(d, np.inf)
    i, j = np.nonzero(d <= radius_m)
    ids = st["station_id"].to_numpy()
    return pd.DataFrame({"station_id": ids[i], "neighbour_id": ids[j], "distance_m": d[i, j].round(1)})

def build_pairs_300m():
    nb = pd.read_csv(NEIGH)
    allp = pairs_within(nb, 300)
    got = allp.groupby("station_id").size().reindex(nb["station_id"].to_numpy(), fill_value=0).to_numpy()
    wrong = int((got != nb["n_within_300m"].to_numpy()).sum())
    assert wrong == 0, f"pair table disagrees with n_within_300m for {wrong} stations"
    print(f"  check OK: pair counts equal n_within_300m for all {len(nb):,} stations of the list")
    ok = set(panel_stations["station_id"])
    return allp[allp["station_id"].isin(ok) & allp["neighbour_id"].isin(ok)].reset_index(drop=True)

station_pairs_300m = cached("station_pairs_300m", build_pairs_300m, sample_dependent=False)
has_nb = station_pairs_300m["station_id"].nunique()
print(f"{len(station_pairs_300m):,} directed pairs; {has_nb:,} of {len(panel_stations):,} panel stations "
      f"({100*has_nb/len(panel_stations):.1f}%) have at least one panel neighbour within 300 m")

## 0.6 One pass over the 25 M rows

The only full scan of the row-level table for the main analysis: counts per **station x UTC hour**. Everything after this (per station, per hour of day, per day, per commune, per weekday) is a `groupby` on this 2-million-row table, and counts add up exactly, so shares are always computed as `sum(count) / sum(n)`.

Columns: `n` = panel captures in that hour; `n_cannot_rent` / `n_cannot_return` = captures where the station cannot rent / take a return; `n_empty`, `n_full`, `n_oos` = captures by `state`; `*_oos` = failures that come from an out-of-service station (not from an empty or full one).

In [ ]:
def build_station_hour():
    load_status(["fetched_at", "station_id", "state", "can_rent", "can_return"], name="status_panel")
    return con.execute("""
        SELECT station_id,
               CAST(fetched_at AS DATE)                         AS date_utc,
               date_part('hour', fetched_at)::INT               AS hour_utc,
               count(*)::INT                                    AS n,
               sum((NOT can_rent)::INT)::INT                    AS n_cannot_rent,
               sum((NOT can_return)::INT)::INT                  AS n_cannot_return,
               sum((NOT can_rent AND state = 'out_of_service')::INT)::INT   AS n_cannot_rent_oos,
               sum((NOT can_return AND state = 'out_of_service')::INT)::INT AS n_cannot_return_oos,
               sum((state = 'empty')::INT)::INT                 AS n_empty,
               sum((state = 'full')::INT)::INT                  AS n_full,
               sum((state = 'out_of_service')::INT)::INT        AS n_oos
        FROM status_panel
        GROUP BY 1, 2, 3
    """).df()

station_hour = cached("station_hour", build_station_hour)
print(f"panel rows covered: {int(station_hour['n'].sum()):,}"
      + (" (expected 24,740,954 on the full period)" if not SAMPLE_DAYS else " (sample days only)"))

COUNTS = ["n", "n_cannot_rent", "n_cannot_return", "n_cannot_rent_oos", "n_cannot_return_oos", "n_empty", "n_full", "n_oos"]

def add_local_time(df):
    # Paris time, derived from the UTC hour (offsets are whole hours, so this is exact)
    ts = (pd.to_datetime(df["date_utc"]) + pd.to_timedelta(df["hour_utc"], unit="h")).dt.tz_localize("UTC")
    loc = ts.dt.tz_convert("Europe/Paris")
    df = df.copy()
    df["hour_paris"], df["weekday_paris"], df["date_paris"] = loc.dt.hour, loc.dt.day_of_week, loc.dt.date
    return df

def shares(df, by, extra=()):
    # sum the counts, then divide: exact weighted shares, in percent
    cols = COUNTS + [c for c in extra if c not in COUNTS]
    g = df.groupby(by, as_index=False)[cols].sum()
    g["pct_cannot_rent"] = 100 * g["n_cannot_rent"] / g["n"]
    g["pct_cannot_return"] = 100 * g["n_cannot_return"] / g["n"]
    return g

## 0.7 Failures with and without an alternative within 300 m

For every panel capture where a station **cannot rent**, is there another panel station within 300 m that **can rent in the same capture**? Same for **cannot take a return**. A neighbour missing from a capture counts as "cannot serve", a station with no neighbour is without alternative by construction. The join is set-based in DuckDB and runs one day at a time, so memory stays small.

Output (station x UTC hour): `n_cannot_rent_alt` = failures that had an alternative (so failures without one = `n_cannot_rent - n_cannot_rent_alt`), same for return. Rows exist only where there was a failure.

In [ ]:
def build_station_hour_alt(pairs=None):
    # pairs: the neighbour table to use (default the 300 m one; 7.4 passes 200 m and 500 m)
    pairs = station_pairs_300m if pairs is None else pairs
    con.register("pairs", pairs[["station_id", "neighbour_id"]])
    files = day_files(SAMPLE_DAYS) if SAMPLE_DAYS else sorted((CLEAN_DIR).glob("date=*/part-0.parquet"))
    out, t0 = [], time.time()
    for k, f in enumerate(files, 1):
        f = Path(f).as_posix()
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE day AS
            SELECT fetched_at, station_id, can_rent, can_return FROM read_parquet('{f}') WHERE in_panel""")
        out.append(con.execute("""
            WITH fail AS (
                SELECT station_id, fetched_at, NOT can_rent AS f_rent, NOT can_return AS f_ret
                FROM day WHERE NOT can_rent OR NOT can_return),
            alt AS (
                SELECT f.station_id, f.fetched_at,
                       coalesce(bool_or(d.can_rent),   false) AS alt_rent,
                       coalesce(bool_or(d.can_return), false) AS alt_ret
                FROM fail f
                LEFT JOIN pairs p ON p.station_id = f.station_id
                LEFT JOIN day d   ON d.station_id = p.neighbour_id AND d.fetched_at = f.fetched_at
                GROUP BY f.station_id, f.fetched_at)
            SELECT f.station_id,
                   CAST(f.fetched_at AS DATE)             AS date_utc,
                   date_part('hour', f.fetched_at)::INT   AS hour_utc,
                   sum((f.f_rent AND a.alt_rent)::INT)::INT AS n_cannot_rent_alt,
                   sum((f.f_ret  AND a.alt_ret )::INT)::INT AS n_cannot_return_alt
            FROM fail f JOIN alt a USING (station_id, fetched_at)
            GROUP BY 1, 2, 3""").df())
        if k % 10 == 0 or k == len(files):
            print(f"  {k}/{len(files)} days ({time.time()-t0:.0f} s)", end="\r")
    con.execute("DROP TABLE IF EXISTS day")
    return pd.concat(out, ignore_index=True)

station_hour_alt = cached("station_hour_alt", build_station_hour_alt)

# one table for the analysis: counts plus failures that had an alternative
sh = station_hour.merge(station_hour_alt, on=["station_id", "date_utc", "hour_utc"], how="left")
sh[["n_cannot_rent_alt", "n_cannot_return_alt"]] = sh[["n_cannot_rent_alt", "n_cannot_return_alt"]].fillna(0).astype(int)
ALT = ["n_cannot_rent_alt", "n_cannot_return_alt"]

def shares_alt(df, by):
    g = shares(df, by, extra=ALT)
    g["pct_cannot_rent_no_alt"] = 100 * (g["n_cannot_rent"] - g["n_cannot_rent_alt"]) / g["n"]
    g["pct_cannot_return_no_alt"] = 100 * (g["n_cannot_return"] - g["n_cannot_return_alt"]) / g["n"]
    return g

## 0.8 Events: demonstrations, closures, damage

`data/external/events.csv` (draft provided, sources in `data/external/SOURCES.md`) lists the events of the period that we know from independent sources: demonstration days, the days Vélib' closed about 110 stations at the prefecture's request, the day of the main damage, the Pope's visit. They are used three ways: to **validate** the measure (section 6: does it see what we know happened?), to run the analysis **with and without** the days flagged `exclude_from_baseline`, and as context in the replay. Every row has a URL, and `status` stays `to_verify` until someone has opened the source and checked it.

In [ ]:
EVENTS_CSV = EXT / "events.csv"
events, EVENT_DAYS = None, set()
if EVENTS_CSV.exists():
    events = pd.read_csv(EVENTS_CSV, parse_dates=["date_start", "date_end"], dtype={"exclude_from_baseline": str})
    events["exclude_from_baseline"] = events["exclude_from_baseline"].str.lower().isin(["true", "1", "yes"])
    for r in events.itertuples():
        if r.exclude_from_baseline:
            EVENT_DAYS |= set(pd.date_range(r.date_start, r.date_end).date)
    todo = int((events["status"] != "verified").sum())
    print(f"{len(events)} events loaded, {len(EVENT_DAYS)} days flagged exclude_from_baseline; "
          f"{todo} still marked to_verify" + (" (check the sources before they go in the report)" if todo else ""))
else:
    print("TODO: data/external/events.csv does not exist yet (section 0.8).")

# Part 1. The story, from the question to the conclusion

Each section below is one section of the report. Run Part 0 first. Figures are saved in `outputs/` (names in the table at the top); the **report figures** are marked below, the others are for the notebook and the slides.

## 1. The question, and how it changed

**Question.** How often does a Vélib' station fail its users (no bike to rent, or no free dock to return) while no other station within 300 m can serve them, and does this differ between Paris and the surrounding communes?

**Defined terms** (same rules for everyone in the group; details in `DATA_DICTIONARY.md`, section 11):

| Term | Rule |
|---|---|
| Cannot rent | `can_rent` false: station not installed or not renting, or 0 bikes |
| Cannot take a return | `can_return` false: station not installed or not returning, or 0 free docks |
| Out of service | not installed, or neither renting nor returning |
| Failure episode | the same side failing in at least 2 consecutive captures (10 minutes) |
| Viable alternative | another panel station within 300 m that is not failing, on the same side, in the same capture |
| Isolated station | no other panel station within 300 m |

Two sides are always kept apart (a person renting needs a bike, a person returning needs a free dock). Distance is station to station, in a straight line: user positions are not observable, so we measure the stations, not the people. "Everywhere" means Paris against the rest of the network (communes). Source for the 300 m: Vélib' Métropole, page "Paris" (https://www.velib-metropole.fr/communes/paris) (accessed 11/10). It states a station every 300 m in Paris (Paris only).

**What changed after each round of feedback** (reflection required by the brief; each row says what we did and where to find it):

| When | What was raised | What we did | Where |
|---|---|---|---|
| Proposal (16.09) to feedback (26.09) | "Failure" and "isolated" not operationalised; too many sources; sampling interval to justify | One precise question; station-to-station 300 m rule; kept only commune boundaries and population; dropped bike counters, commuting, weather and school calendar | 1, 5 |
| Peer review 1 (on the 29.09 submission) | `last_reported` read as a measurement time; usable for whom; unit of comparison; population is not users; station list changes; truncated capture not detected | `last_reported` is an hourly station signal: all rows kept, no deduplication; rent and return separate; Paris vs rest; population as context only; same-day station list and flags; content check added to the collector | 3, 5 |
| Peer review 2 (on the 03.11 submission) | collection frequency; one-way station check; `MIN_STATIONS` unjustified, "loses up to 92% of data"; flat table, redundancy | Sampling test on our own archive; **92% not reproduced: the rule removes whole captures, never stations, and removed 0 rows on the full period**; absent station-captures counted (about 770 of 24.9 M, negligible); flat on purpose, normalised alternative benchmarked | 3, 4, 5 |

Observation versus interpretation: this section is the only one without data; everything else states what the data show first and what we make of it second.

## 2. Data and collection

**Source.** Vélib' Métropole GBFS feed (`station_status` every 5 minutes, `station_information` every hour), operated by Smovengo, redistributed under ODbL; no personal data. We collected it ourselves on two always-on cloud VMs (VM1 main series, VM2 reliability check) from 20.09 to 16.11.2026; the feed keeps no history, so a missed capture is lost. Raw files are kept untouched on Zenodo (DOI 10.5281/zenodo.24444738); the collection scripts and the VM set-up guide are on GitHub.

**What could bias the data.** The two VMs ran in different Azure regions (Denmark East, Sweden Central), so a regional outage would not have hit both; the period is eight autumn weeks with unusual events (section 6), so it does not represent a whole year; a capture is the state when we asked, not an instant.

**Figure (notebook / slides): captures per day.** Every full day has 288 captures and no request failed.

In [ ]:
def small_table(name):
    p = locate(name, required=False)
    if p is None:
        print(f"{name} not found under data/vm1 (it comes from vm1_csv_logs.tar.gz): this figure is skipped.")
    return p

def col(df, key):
    # find a column by a part of its name (headers contain symbols such as %)
    return next(c for c in df.columns if key in c)

p = small_table("coverage_status.csv")
if p:
    cov = pd.read_csv(p)
    cov["date"] = pd.to_datetime(cov["date"])
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.bar(cov["date"], cov["captured"], width=0.8, color="#2a9d8f", label="captures received")
    ax.axhline(288, color="#444", lw=1, ls="--", label="288 expected per day")
    ax.set_ylabel("status captures per day")
    ax.set_title("Collection coverage, VM1 (20.09 is a partial day)")
    ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(OUT / "fig_coverage.png", dpi=150); plt.show()
    full = cov[cov["captured"] >= 288]
    print(f"{len(cov)} days, {int(cov['captured'].sum()):,} captures, failed requests: {int((cov['attempted'] - cov['captured']).sum())}, "
          f"full days with at least 288 captures: {len(full)}")

## 3. Quality and cleaning

**Principle: flag, do not alter.** Nothing in the feed can correct a count, so no count is corrected. A row is removed only if it carries no information (truncated capture below 1,400 stations, exact duplicate): on the full period these rules removed **0 rows**. Everything else becomes a flag, and `in_panel` is the single selector for the analysis. Reproducible with `code/run_all.py`.

**Figure (notebook / slides): share of rows flagged by each rule.** The rules and thresholds are in `cleaning_log.csv` (silent: older than 180 minutes; frozen: unchanged counts for 12 hours or more while in service; event stations: capacity 100 or more).

**Station-captures absent** (a listed station with no row in a capture). Among the regular stations (not temporary, not event stations), 4 station-days are incomplete (Bastille - Bourdon and Caire - Dussoubs, on 20 and 21 October). This is 770 missing captures out of 24.9 million, so 0.003 %. This is negligible, so we keep these stations in the panel. Temporary and event stations also have incomplete days, but they are not in the panel. The cell after the figure recomputes these numbers.

In [ ]:
p = small_table("cleaning_log.csv")
if p:
    cl = pd.read_csv(p)
    pct = cl[cl["step"] != "in_panel"].copy()
    pct["pct_flagged"] = 100 * pct["rows_flagged"] / pct["rows_before"]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(pct["step"], pct["pct_flagged"], color="#6c757d")
    for y, v in enumerate(pct["pct_flagged"]):
        ax.text(v, y, f" {v:.2f}%", va="center", fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel("% of the rows of the cleaned table")
    ax.set_title("What each cleaning rule flagged (nothing was dropped)")
    fig.tight_layout(); fig.savefig(OUT / "fig_cleaning.png", dpi=150); plt.show()
    sel = cl[cl["step"] == "in_panel"]["rows_flagged"].iloc[0]
    print(f"rows removed by any rule: {int((cl['rows_before'] - cl['rows_after']).sum())}; rows selected in the panel: {int(sel):,}")
    display(cl)

In [ ]:
# Absent station-captures: expected captures of the day minus captures of the station that day
p_sd, p_cq = small_table("stations_daily.csv"), small_table("capture_quality.csv")
if p_sd and p_cq:
    sd = pd.read_csv(p_sd)
    cq = pd.read_csv(p_cq)
    cq["date"] = pd.to_datetime(cq["fetched_at"], utc=True).dt.strftime("%Y-%m-%d")
    per_day = cq.groupby("date").size().rename("captures_that_day")
    x = sd.merge(per_day, on="date", how="left")
    x["absent"] = x["captures_that_day"] - x["captures"]
    regular = x[~x["temporary"] & ~x["event_station"]]
    for label, g in [("regular stations", regular), ("all stations", x)]:
        gap = g[g["absent"] > 0]
        print(f"{label}: {len(gap)} incomplete station-days, {int(gap['absent'].sum()):,} absent captures, "
              f"{100 * gap['absent'].sum() / g['captures_that_day'].sum():.4f} % of the expected captures")
    display(regular[regular["absent"] > 0][["date", "station_id", "name", "captures", "captures_that_day", "absent"]])


## 4. Storage

The brief asks for an explicit storage decision, justified by the data and the intended queries, with a detailed comparison with at least one other solution. The **benchmark** below builds the same table in several formats and times the same four queries on each. It is switched off by default (`RUN_BENCHMARK = False`): it is slow and needs disk. Try it first with `SAMPLE_DAYS` on two or three days.

- Table: the cleaned table, 16 columns (the same for every format, including the station attributes repeated on every row: name, lat, lon, capacity, code), all rows (`in_panel` kept as a column).
- Formats: CSV, CSV gzip, NDJSON, Parquet (snappy, zstd, zstd sorted by station), DuckDB file, SQLite flat (indexed), SQLite normalised (station table + status table, indexed).
- Queries: **q1** share of time each station cannot rent (full scan, 3 columns); **q2** hourly failure profile (group by hour); **q3** one station's whole timeline (point lookup over all days); **q4** the whole network at one capture (point-in-time lookup).
- Each query runs 3 times; the table gives the first run and the median. The first run is not a guaranteed cold cache: say so in the report.
- **Report the machine** (printed below), because different laptops (yours, Lara's, a teammate's) will not give the same times. Peak disk use is the largest single format (NDJSON, several GB on the full period); each format is deleted after it is measured unless `keep_files=True`.
- Normalisation caveat: the normalised SQLite keeps one row per station for the attributes, so it drops their day-to-day changes. State it when you compare sizes.
- PostgreSQL or another server database is **not** benchmarked: one analyst, one batch load, read-only analysis, no concurrency. Say so in one paragraph.

In [ ]:
BENCH_DIR = DATA / "bench"
BENCH_COLS = ["fetched_at", "station_id", "stationCode", "name", "lat", "lon", "capacity",
              "num_bikes_available", "num_docks_available", "is_installed", "is_renting", "is_returning",
              "state", "can_rent", "can_return", "in_panel"]
BENCH_FORMATS = ["csv", "csv_gz", "ndjson", "parquet_snappy", "parquet_zstd", "parquet_zstd_sorted_by_station",
                 "duckdb_file", "sqlite_flat", "sqlite_normalised"]

def _size(p):
    p = Path(p)
    return p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def _machine():
    info = {"platform": platform.platform(), "processor": platform.processor(), "cpu_count": os.cpu_count(),
            "python": platform.python_version(), "duckdb": duckdb.__version__, "pandas": pd.__version__,
            "sqlite": sqlite3.sqlite_version}
    try:
        import psutil
        info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        info["ram_gb"] = "unknown (pip install psutil to record it)"
    return info

def run_storage_benchmark(formats=None, repeats=3, keep_files=False):
    formats = formats or BENCH_FORMATS
    shutil.rmtree(BENCH_DIR, ignore_errors=True)
    BENCH_DIR.mkdir(parents=True)
    load_status(BENCH_COLS, panel_only=False, name="bench_src")
    cols = ", ".join(BENCH_COLS)
    sel = f"SELECT {cols}, CAST(fetched_at AS DATE) AS date FROM bench_src"

    # one station and one capture for the lookup queries (chosen from the data, not by hand)
    sid = int(panel_stations["station_id"].iloc[len(panel_stations) // 2])
    lo, hi = con.execute("SELECT min(fetched_at), max(fetched_at) FROM bench_src").fetchall()[0]
    mid = lo + (hi - lo) / 2
    t = con.execute("SELECT fetched_at FROM bench_src WHERE station_id = ? ORDER BY abs(epoch(fetched_at) - epoch(?)) LIMIT 1",
                    [sid, mid]).fetchall()[0][0]
    t_iso = pd.Timestamp(t).strftime("%Y-%m-%d %H:%M:%S")
    print(f"machine: {_machine()}\nlookup station: {sid}   lookup capture: {t_iso}")

    duck_q = {
        "q1_share_cannot_rent_per_station": "SELECT station_id, avg((NOT can_rent)::INT) FROM {src} WHERE in_panel GROUP BY station_id",
        "q2_hourly_profile": "SELECT date_part('hour', fetched_at) AS h, avg((NOT can_rent)::INT), avg((NOT can_return)::INT) FROM {src} WHERE in_panel GROUP BY h ORDER BY h",
        "q3_one_station_timeline": f"SELECT fetched_at, num_bikes_available, num_docks_available, state FROM {{src}} WHERE station_id = {sid} ORDER BY fetched_at",
        "q4_network_at_one_capture": f"SELECT station_id, num_bikes_available, num_docks_available, state FROM {{src}} WHERE fetched_at = TIMESTAMP '{t_iso}'",
    }
    sqlite_q = {
        "q1_share_cannot_rent_per_station": ("SELECT station_id, avg(1 - can_rent) FROM status WHERE in_panel = 1 GROUP BY station_id", ()),
        "q2_hourly_profile": ("SELECT CAST(substr(fetched_at, 12, 2) AS INT) AS h, avg(1 - can_rent), avg(1 - can_return) FROM status WHERE in_panel = 1 GROUP BY h ORDER BY h", ()),
        "q3_one_station_timeline": ("SELECT fetched_at, num_bikes_available, num_docks_available, state FROM status WHERE station_id = ? ORDER BY fetched_at", (sid,)),
        "q4_network_at_one_capture": ("SELECT station_id, num_bikes_available, num_docks_available, state FROM status WHERE fetched_at = ?", (t_iso,)),
    }

    def build_sqlite(path, normalised):
        db = sqlite3.connect(path)
        status_cols = [c for c in BENCH_COLS if not (normalised and c in ("stationCode", "name", "lat", "lon", "capacity"))]
        if normalised:
            db.execute("CREATE TABLE station (station_id INTEGER PRIMARY KEY, stationCode TEXT, name TEXT, lat REAL, lon REAL, capacity INTEGER)")
            rows = con.execute("SELECT station_id, any_value(stationCode), any_value(name), any_value(lat), any_value(lon), max(capacity) FROM bench_src GROUP BY station_id").fetchall()
            db.executemany("INSERT INTO station VALUES (?,?,?,?,?,?)", rows)
        db.execute("CREATE TABLE status (" + ", ".join(
            f"{c} " + ("TEXT" if c == "fetched_at" else "REAL" if c == "lat" or c == "lon" else "TEXT" if c in ("stationCode", "name", "state") else "INTEGER")
            for c in status_cols) + ")")
        pick = ", ".join("strftime(fetched_at, '%Y-%m-%d %H:%M:%S') AS fetched_at" if c == "fetched_at" else c for c in status_cols)
        res = con.execute(f"SELECT {pick} FROM bench_src")
        ins = f"INSERT INTO status VALUES ({','.join('?' * len(status_cols))})"
        while True:
            rows = res.fetchmany(200_000)
            if not rows:
                break
            db.executemany(ins, rows)
        db.execute("CREATE INDEX ix_station_time ON status(station_id, fetched_at)")
        db.execute("CREATE INDEX ix_time ON status(fetched_at)")
        db.commit()
        db.close()

    def duck_source(fmt, path):
        p = Path(path).as_posix()
        return {"csv": f"read_csv('{p}', header = true)", "csv_gz": f"read_csv('{p}', header = true)",
                "ndjson": f"read_json('{p}', format = 'newline_delimited')",
                "parquet_snappy": f"read_parquet('{p}/**/*.parquet', hive_partitioning = true)",
                "parquet_zstd": f"read_parquet('{p}/**/*.parquet', hive_partitioning = true)",
                "parquet_zstd_sorted_by_station": f"read_parquet('{p}/**/*.parquet', hive_partitioning = true)",
                "duckdb_file": "bdb.status"}[fmt]

    rows = []
    for fmt in formats:
        path = BENCH_DIR / {"csv": "status.csv", "csv_gz": "status.csv.gz", "ndjson": "status.ndjson",
                            "duckdb_file": "status.duckdb", "sqlite_flat": "status_flat.sqlite",
                            "sqlite_normalised": "status_norm.sqlite"}.get(fmt, fmt)
        print(f"\n== {fmt}")
        t0 = time.perf_counter()
        if fmt == "csv":
            con.execute(f"COPY ({sel}) TO '{path.as_posix()}' (FORMAT CSV, HEADER)")
        elif fmt == "csv_gz":
            con.execute(f"COPY ({sel}) TO '{path.as_posix()}' (FORMAT CSV, HEADER, COMPRESSION gzip)")
        elif fmt == "ndjson":
            con.execute(f"COPY ({sel}) TO '{path.as_posix()}' (FORMAT JSON)")
        elif fmt.startswith("parquet"):
            comp = "SNAPPY" if fmt == "parquet_snappy" else "ZSTD"
            order = " ORDER BY station_id, fetched_at" if fmt.endswith("sorted_by_station") else ""
            con.execute(f"COPY ({sel}{order}) TO '{path.as_posix()}' (FORMAT PARQUET, PARTITION_BY (date), COMPRESSION {comp}, OVERWRITE_OR_IGNORE)")
        elif fmt == "duckdb_file":
            con.execute(f"ATTACH '{path.as_posix()}' AS bdb")
            con.execute(f"CREATE TABLE bdb.status AS SELECT {cols} FROM bench_src")
            con.execute("DETACH bdb")
            con.execute(f"ATTACH '{path.as_posix()}' AS bdb (READ_ONLY)")
        elif fmt in ("sqlite_flat", "sqlite_normalised"):
            build_sqlite(path, normalised=(fmt == "sqlite_normalised"))
        else:
            raise ValueError(fmt)
        build_s = time.perf_counter() - t0
        row = {"format": fmt, "size_mb": round(_size(path) / 1e6, 1), "build_s": round(build_s, 1)}
        print(f"  size {row['size_mb']:,} MB, build {build_s:.0f} s")

        if fmt.startswith("sqlite"):
            db = sqlite3.connect(path)
            if fmt == "sqlite_normalised":
                pass   # q1-q4 only use status columns, no join needed
            run = lambda name: db.execute(*sqlite_q[name]).fetchall()
        else:
            src = duck_source(fmt, path)
            run = lambda name: con.execute(duck_q[name].format(src=src)).fetchall()
        for name in duck_q:
            times = []
            for _ in range(repeats):
                t0 = time.perf_counter()
                res = run(name)
                times.append(time.perf_counter() - t0)
            row[name.split("_")[0] + "_first_s"] = round(times[0], 3)
            row[name.split("_")[0] + "_median_s"] = round(statistics.median(times), 3)
            print(f"  {name}: first {times[0]:.3f} s, median {statistics.median(times):.3f} s ({len(res):,} rows returned)")
        if fmt.startswith("sqlite"):
            db.close()
        if fmt == "duckdb_file":
            con.execute("DETACH bdb")
        rows.append(row)
        if not keep_files:
            shutil.rmtree(path, ignore_errors=True) if Path(path).is_dir() else Path(path).unlink(missing_ok=True)

    res = pd.DataFrame(rows)
    res.to_csv(OUT / "storage_benchmark.csv", index=False)
    (OUT / "storage_benchmark_machine.json").write_text(json.dumps(
        {"machine": _machine(), "sample_days": SAMPLE_DAYS, "rows": int(con.execute("SELECT count(*) FROM bench_src").fetchall()[0][0])}, indent=2))
    if not keep_files:
        shutil.rmtree(BENCH_DIR, ignore_errors=True)
    return res

if RUN_BENCHMARK:
    benchmark = run_storage_benchmark()
    display(benchmark)
else:
    print("Benchmark switched off (RUN_BENCHMARK = False). Set it to True on purpose; start with SAMPLE_DAYS.")

**Storage decision** (written 23.11, from the benchmark on the full period: 24,998,625 rows, 58 days; Windows 11 laptop, 12 CPU threads, 16.9 GB of RAM, DuckDB 2.0.1, SQLite 3.49.1, DuckDB limited to 2 GB and 2 threads; figures in `outputs/storage_benchmark.csv` and `outputs/storage_design_benchmark.csv`).

**What we can see:**

- **Size.** The same table takes 3,546 MB in CSV, 9,021 MB in NDJSON and 4,061 MB in SQLite (with its indexes), against 28 MB in Parquet with zstd (38 MB with snappy), which is about 125 times less than the CSV. The DuckDB file is 300 MB. CSV compressed with gzip is 100 MB, but it is slow to read.
- **Scans.** For the share of time each station cannot rent (q1) the median is 0.6 to 0.8 s on Parquet and on the DuckDB file, against 12 to 26 s on SQLite and 20 to 38 s on the text files. For the hourly profile (q2) it is 1.5 to 3.3 s on Parquet and 2.7 s on the DuckDB file, against 37 s on SQLite. Parquet is more than 10 times faster than SQLite on every scan.
- **Lookups.** SQLite is the fastest for one station (q3, 0.04 s) and for one capture (q4, 0.02 s), but Parquet is not far (0.07 to 0.22 s and 0.04 to 0.08 s) and the DuckDB file is the best on q4 (0.01 s).
- **Small choices inside Parquet.** Sorting by station did not help (29 MB and 0.70 s for q1, against 28 MB and 0.77 s). Snappy reads faster than zstd on q2 (1.5 s against 3.3 s) but the file is 10 MB bigger.
- **Design A against design B.** Design A is Parquet partitioned by date with DuckDB, design B is SQLite with indexes. Initial load: 33 s against 234 s. Adding 1 day: 4.3 s against 19 s. Total size: 24 MB against 2,762 MB, of which 1,548 MB are indexes. One week of data: 0.09 s against 22.8 s. Paris against the rest: 1.45 s against 21.9 s. Event days against other days: 1.03 s against 32.8 s. The 2 engines return the same numbers for these 3 queries, so the times can be compared.

**What we think:**

Our data is an archive that is written once, read by 1 analyst, and used mostly for scans over a few columns, with only a few lookups. The numbers fit this: Parquet is about 125 times smaller than the CSV, more than 10 times faster than SQLite on the scans, and only a few hundredths of a second slower on a lookup. So we choose Parquet partitioned by date, queried with DuckDB, and we publish it on Zenodo with the raw files. CSV or NDJSON would only be readable copies. We keep zstd, because the size matters for sharing and 3 s is fine for 1 analyst.

SQLite becomes the better choice for many lookups on single stations, but here the gain is a few milliseconds. Normalising SQLite saves 1.3 GB (4,061 MB to 2,753 MB), but it keeps only 1 row of attributes per station and loses their day-to-day changes. In Parquet the repeated attributes (name, position, capacity) cost nothing, because the compression removes the repetition. A server database such as PostgreSQL was not tested: 1 analyst, 1 batch load, read-only analysis and no concurrency, so it would only add administration.

**Limits:**

The times come from 1 laptop and depend on it. The first run of each query is not guaranteed to be a cold cache, and each query was measured only 3 times. The 22.8 s of SQLite on the 1-week query is slow for a table with a time index: we did not tune this query, so a better written one could be faster. The choice applies to this workload: a live service with several writers and many lookups per station would point elsewhere.

**Second run** (23.11, Windows 11 laptop, 20 CPU threads, 64 GB of RAM, DuckDB 2.0.1, SQLite 3.45.3, DuckDB limited to 2 GB and 2 threads): the sizes are the same (CSV 3,546 MB, NDJSON 9,021 MB, SQLite 4,061 MB, Parquet zstd 28 MB, design A 24 MB against design B 2,762 MB) and the ranking is the same: Parquet is much faster than SQLite on the scans (q1 0.35 s against 9.1 s, q2 1.6 s against 21.5 s) and SQLite is the fastest for one station (q3 0.03 s against 0.13 s). Only the seconds change from one machine to the other, which is why we rely on the sizes and the ranking.

In [ ]:
# Design A (Parquet fact + small reference files, joined in DuckDB) against Design B (SQLite, fact + reference tables).
# Same logical model in both: a fact table without station attributes, a station table, a commune table (is_paris),
# a table of event days. Needs section 0.4 (communes) and 0.8 (events) for the two join queries; needs at least 2 days.
def run_design_benchmark(repeats=3, keep_files=False):
    D = DATA / "bench_design"
    shutil.rmtree(D, ignore_errors=True); D.mkdir(parents=True)
    fact = ["fetched_at", "station_id", "num_bikes_available", "num_docks_available", "is_installed", "is_renting",
            "is_returning", "state", "can_rent", "can_return", "in_panel"]
    load_status(fact + ["stationCode", "name", "lat", "lon", "capacity"], panel_only=False, name="bench_src")
    days = [r[0] for r in con.execute("SELECT DISTINCT CAST(fetched_at AS DATE) FROM bench_src ORDER BY 1").fetchall()]
    if len(days) < 2:
        print("Needs at least 2 days (the last day is appended to measure incremental loading)."); return None
    last, first_days = days[-1], days[:-1]
    d0, d1 = days[0], days[min(7, len(days) - 1)]           # time-range query: first week (or what exists)
    sel = f"SELECT {', '.join(fact)}, CAST(fetched_at AS DATE) AS date FROM bench_src"
    have_c, have_e = communes is not None, bool(EVENT_DAYS)
    print(f"machine: {_machine()}\ndays: {len(days)}; the last one ({last}) is appended after the initial load")

    # reference tables (same content in both designs)
    st_ref = con.execute("SELECT station_id, any_value(stationCode) AS stationCode, any_value(name) AS name, any_value(lat) AS lat, any_value(lon) AS lon, max(capacity) AS capacity FROM bench_src GROUP BY station_id").df()
    com = communes[["station_id", "commune", "is_paris"]].assign(is_paris=lambda d: d["is_paris"].astype(int)) if have_c else None
    ev = pd.DataFrame({"date": sorted(EVENT_DAYS)}) if have_e else None
    out = {"A_parquet_duckdb": {}, "B_sqlite": {}}

    # ---------------- Design A
    A = D / "A"; (A / "fact").mkdir(parents=True)
    t0 = time.perf_counter()
    con.execute(f"COPY ({sel} WHERE CAST(fetched_at AS DATE) < DATE '{last}') TO '{(A / 'fact').as_posix()}' (FORMAT PARQUET, PARTITION_BY (date), COMPRESSION ZSTD, OVERWRITE_OR_IGNORE)")
    st_ref.to_parquet(A / "station.parquet")
    if have_c: com.to_parquet(A / "commune.parquet")
    if have_e: ev.assign(date=lambda d: pd.to_datetime(d["date"])).to_parquet(A / "event_day.parquet")
    out["A_parquet_duckdb"]["initial_load_s"] = round(time.perf_counter() - t0, 2)
    t0 = time.perf_counter()
    con.execute(f"COPY ({sel} WHERE CAST(fetched_at AS DATE) = DATE '{last}') TO '{(A / 'fact').as_posix()}' (FORMAT PARQUET, PARTITION_BY (date), COMPRESSION ZSTD, OVERWRITE_OR_IGNORE)")
    out["A_parquet_duckdb"]["append_one_day_s"] = round(time.perf_counter() - t0, 2)
    out["A_parquet_duckdb"]["size_mb"] = round(_size(A) / 1e6, 1)
    out["A_parquet_duckdb"]["n_files"] = sum(1 for f in A.rglob("*") if f.is_file())
    out["A_parquet_duckdb"]["index_overhead_mb"] = 0.0

    # ---------------- Design B
    Bp = D / "B.sqlite"
    db = sqlite3.connect(Bp)
    t0 = time.perf_counter()
    typ = lambda c: "TEXT" if c in ("fetched_at", "state") else "INTEGER"
    db.execute("CREATE TABLE status (" + ", ".join(f"{c} {typ(c)}" for c in fact) + ")")
    db.execute("CREATE TABLE station (station_id INTEGER PRIMARY KEY, stationCode TEXT, name TEXT, lat REAL, lon REAL, capacity INTEGER)")
    db.executemany("INSERT INTO station VALUES (?,?,?,?,?,?)", st_ref.itertuples(index=False, name=None))
    if have_c:
        db.execute("CREATE TABLE commune (station_id INTEGER PRIMARY KEY, commune TEXT, is_paris INTEGER)")
        db.executemany("INSERT INTO commune VALUES (?,?,?)", com.itertuples(index=False, name=None))
    if have_e:
        db.execute("CREATE TABLE event_day (date TEXT PRIMARY KEY)")
        db.executemany("INSERT INTO event_day VALUES (?)", [(str(d),) for d in ev["date"]])
    pick = ", ".join("strftime(fetched_at, '%Y-%m-%d %H:%M:%S') AS fetched_at" if c == "fetched_at" else c for c in fact)
    ins = f"INSERT INTO status VALUES ({','.join('?' * len(fact))})"
    def load_days(where):
        res = con.execute(f"SELECT {pick} FROM bench_src WHERE {where}")
        while True:
            rows = res.fetchmany(200_000)
            if not rows: break
            db.executemany(ins, rows)
    load_days(f"CAST(fetched_at AS DATE) < DATE '{last}'"); db.commit()
    size_noidx = Bp.stat().st_size
    db.execute("CREATE INDEX ix_station_time ON status(station_id, fetched_at)")
    db.execute("CREATE INDEX ix_time ON status(fetched_at)"); db.commit()
    out["B_sqlite"]["initial_load_s"] = round(time.perf_counter() - t0, 2)
    size_idx = Bp.stat().st_size
    t0 = time.perf_counter()
    load_days(f"CAST(fetched_at AS DATE) = DATE '{last}'"); db.commit()
    out["B_sqlite"]["append_one_day_s"] = round(time.perf_counter() - t0, 2)
    out["B_sqlite"]["size_mb"] = round(Bp.stat().st_size / 1e6, 1)
    out["B_sqlite"]["n_files"] = 1
    out["B_sqlite"]["index_overhead_mb"] = round((size_idx - size_noidx) / 1e6, 1)   # measured after the initial load

    # ---------------- queries (same question, two engines); the results must agree
    G = f"read_parquet('{(A / 'fact').as_posix()}/**/*.parquet', hive_partitioning = true)"
    queries = {
        "q5_time_range_first_week": (
            f"SELECT avg((NOT can_rent)::INT) FROM {G} WHERE in_panel AND date >= DATE '{d0}' AND date < DATE '{d1}'",
            "SELECT avg(1 - can_rent) FROM status WHERE in_panel = 1 AND fetched_at >= ? AND fetched_at < ?", (f"{d0} 00:00:00", f"{d1} 00:00:00")),
    }
    if have_c:
        queries["q6_paris_vs_rest_join"] = (
            f"SELECT c.is_paris, avg((NOT s.can_rent)::INT), avg((NOT s.can_return)::INT) FROM {G} s JOIN read_parquet('{(A / 'commune.parquet').as_posix()}') c USING (station_id) WHERE s.in_panel GROUP BY 1 ORDER BY 1",
            "SELECT c.is_paris, avg(1 - s.can_rent), avg(1 - s.can_return) FROM status s JOIN commune c USING (station_id) WHERE s.in_panel = 1 GROUP BY 1 ORDER BY 1", ())
    if have_e:
        queries["q7_event_days_join"] = (
            f"SELECT (s.date IN (SELECT CAST(date AS DATE) FROM read_parquet('{(A / 'event_day.parquet').as_posix()}'))) AS ev, avg((NOT s.can_rent)::INT) FROM {G} s WHERE s.in_panel GROUP BY 1 ORDER BY 1",
            "SELECT (substr(s.fetched_at, 1, 10) IN (SELECT date FROM event_day)) AS ev, avg(1 - s.can_rent) FROM status s WHERE s.in_panel = 1 GROUP BY 1 ORDER BY 1", ())
    for name, (qa, qb, pb) in queries.items():
        ra = rb = None
        for design, run in (("A_parquet_duckdb", lambda: con.execute(qa).fetchall()), ("B_sqlite", lambda: db.execute(qb, pb).fetchall())):
            ts = []
            for _ in range(repeats):
                t0 = time.perf_counter(); r = run(); ts.append(time.perf_counter() - t0)
            out[design][name.split("_")[0] + "_first_s"] = round(ts[0], 3)
            out[design][name.split("_")[0] + "_median_s"] = round(statistics.median(ts), 3)
            if design.startswith("A"): ra = r
            else: rb = r
        same = np.allclose(np.array(ra, dtype=float), np.array(rb, dtype=float), atol=1e-9)
        print(f"  {name}: results {'agree' if same else 'DIFFER (check the two queries before comparing times)'}")
    db.close()
    res = pd.DataFrame(out).T.rename_axis("design").reset_index()
    res.to_csv(OUT / "storage_design_benchmark.csv", index=False)
    if not keep_files: shutil.rmtree(D, ignore_errors=True)
    if not (have_c and have_e):
        print("Skipped queries:" + ("" if have_c else " q6 (needs station_communes.csv, section 0.4)") + ("" if have_e else " q7 (needs events.csv, section 0.8)"))
    return res

if RUN_BENCHMARK:
    design_benchmark = run_design_benchmark()
    if design_benchmark is not None: display(design_benchmark)
else:
    print("Design benchmark switched off (RUN_BENCHMARK = False).")

**Design A against design B: what is compared and why.** A real project rarely stores one table. We compare two *complete* designs for the same logical model (fact table, station table, commune table, event days):

| | Design A (ours) | Design B (relational alternative) |
|---|---|---|
| Facts | Parquet, partitioned by date | SQLite table `status`, indexed on station+time and on time |
| Reference | `station`, `commune`, `event_day` as small Parquet files | the same three tables in the SQLite file |
| Engine | DuckDB, SQL across files | SQLite, SQL with joins |

Measured: initial load, **appending one new day** (how we actually collected), total size and number of files, **index overhead** (B), and three more queries that come from the project's own question: a one-week time range (partition pruning in A, time index in B), Paris against the rest (fact joined to the commune table) and event days against other days (join to the event table). The two engines must return the same numbers, otherwise the times are not comparable: the cell checks it. Together with q1 to q4 above this covers the workload suggested in the course material (time-range scan, aggregation, selective filter, reference join, geographic grouping, external series, ingestion). **Not benchmarked, on purpose:** a server database (PostgreSQL with PostGIS) and a document store: one analyst, batch loading, read-only analysis, no concurrency. One paragraph of justification, no table.

**Commune contours as GeoParquet** would be the natural way to store a true geographic layer. We store only the result of the spatial join (a commune per station), because the question needs nothing else; mention this as a possible extension if the contours are used for a map.

## 5. Sampling: why every 5 minutes, and what `last_reported` is

**What we tested instead of assuming.** `last_reported` is an hourly station signal, not the moment the counts were measured: the counts change several times an hour under an unchanged `last_reported`. So all rows are kept, no deduplication. To check that 5-minute capture carries information, we kept only one capture every 15, 30 or 60 minutes (`sampling_sensitivity.csv`) and counted again.

**Observation.** The share of time failing barely moves between 60 and 5 minutes, but the episodes found at 5 minutes are only partly found at coarser rates: one capture in 15 minutes keeps about 54% of them, one in 60 minutes about 16%. **Interpretation.** 5 minutes was a defensible choice (a finer rate would cost about five times the storage, and 300 m is about 4 minutes on foot).

**Limits, stated, not hidden.** States shorter than about 10 minutes are invisible (an episode needs 2 consecutive captures), so durations are "time seen failing" (up to one interval longer at each end). A capture is the state when we asked. The feed's real refresh rate is not documented; going below 5 minutes cannot be tested on past data.

In [ ]:
p = small_table("sampling_sensitivity.csv")
if p:
    ss = pd.read_csv(p)
    c_ep, c_seen, c_share = col(ss, "vs_5min"), col(ss, "seen_at_all"), col(ss, "share_of_rows")
    sides = list(ss["side"].unique())
    fig, axes = plt.subplots(1, len(sides), figsize=(5.2 * len(sides), 3.6), squeeze=False)
    for ax, side in zip(axes[0], sides):
        d = ss[ss["side"] == side].reset_index(drop=True)
        x = np.arange(len(d))
        ax.bar(x - 0.2, d[c_ep], width=0.4, color="#2a9d8f", label="episodes kept (% of the 5-minute count)")
        ax.bar(x + 0.2, d[c_seen], width=0.4, color="#edae49", label="5-minute episodes seen at all (%)")
        ax.set_xticks(x); ax.set_xticklabels(d["sampling"].astype(str), rotation=20, ha="right", fontsize=8)
        ax.set_title(f"{side}: share of time failing {d[c_share].min():.2f}% to {d[c_share].max():.2f}%", fontsize=9)
        ax.set_ylabel("%")
    axes[0][0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / "fig_sampling.png", dpi=150); plt.show()
    display(ss)
p = small_table("last_reported_check.csv")
if p:
    display(pd.read_csv(p))

## 6. Validation with events: does the measure see what we know happened?

A good test of a measure is a **known event**: something that happened, dated by independent sources *before* we look at our data, that the measure should detect. Between 20.09 and 16.11 there are several (`events.csv`): the Pope's visit (stations closed in security perimeters, two giant temporary stations), demonstration days, the day of the main damage (17.10), and two windows in which Vélib' closed about 110 stations at the prefecture's request (23 to 26.10 and 30.10 to 2.11).

**What each should look like.** The closures are the clear signal: 110 of 1,515 stations is about 7% of the network, for about three days. The damage of 17.10 (6 stations reported heavily damaged) is far too small to move a network-wide line (under 0.5%) and can only be checked station by station. Closed or emptied stations may appear as out of service or simply as empty, so we look at both. A reader should also see a quiet period for contrast.

**Rules.** The list of events is fixed first, from the sources; we then look at the data. A spike we cannot match to a source is not explained away. We describe, we do not claim causes. If an event is *not* visible, that is also a result: something to understand about the feed or the flags before trusting later numbers.

**Report figure 1: the network over time, with the events.**

In [ ]:
sh_l = add_local_time(sh)                       # adds hour_paris, weekday_paris, date_paris
net = shares_alt(sh_l, ["date_paris", "hour_paris"])
net["ts"] = pd.to_datetime(net["date_paris"]) + pd.to_timedelta(net["hour_paris"], unit="h")
net["pct_oos"] = 100 * net["n_oos"] / net["n"]
net = net.sort_values("ts").reset_index(drop=True)
daily = shares_alt(sh_l, "date_paris")
daily["date"] = pd.to_datetime(daily["date_paris"]) + pd.Timedelta(hours=12)   # plotted at midday of the day
daily["pct_oos"] = 100 * daily["n_oos"] / daily["n"]

KIND_COLORS = {"planned_closure": "#4c78a8", "violence": "#d1495b", "mobilisation": "#edae49", "visit": "#8a8a8a"}

def shade_events(ax, labels=True, only=None):
    if events is None:
        return
    for r in events.itertuples():
        if only and r.kind not in only:
            continue
        ax.axvspan(r.date_start, r.date_end + pd.Timedelta(days=1), color=KIND_COLORS.get(r.kind, "#999"), alpha=0.18, lw=0)
        if labels:
            ax.text(r.date_start + (r.date_end - r.date_start) / 2 + pd.Timedelta(hours=12), 1.0, r.event_id,
                    transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=7)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax = axes[0]
ax.plot(daily["date"], daily["pct_cannot_rent"], label="cannot rent", color="#d1495b")
ax.plot(daily["date"], daily["pct_cannot_return"], label="cannot take a return", color="#edae49")
ax.plot(daily["date"], daily["pct_oos"], label="out of service", color="#6c757d")
ax.set_ylabel("% of station-captures (daily)"); ax.legend(loc="upper left", fontsize=8, ncol=3)
shade_events(ax)
ax.set_title("Share of the network that cannot serve, by day, with known events (shaded)", pad=16)
ax = axes[1]
ax.plot(net["ts"], net["pct_oos"], color="#6c757d", lw=0.8)
ax.set_ylabel("% out of service (hourly)"); ax.set_xlabel("Paris time")
shade_events(ax, labels=False)
if events is not None:
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=c, alpha=0.4, label=k.replace("_", " ")) for k, c in KIND_COLORS.items()], fontsize=7, loc="upper left", ncol=4)
fig.tight_layout(); fig.savefig(OUT / "fig_events_timeline.png", dpi=150); plt.show()
if events is None:
    print("TODO: events.csv missing (section 0.8): the shading and the table below need it.")

**Observation: how far above the noise?** For each event, the peak of the hourly share inside its window against the hours outside any event window (baseline = all hours outside every event window, and more than one day away from the major disruptions flagged `exclude_from_baseline`: the Pope's visit, 17 Oct and the two closures; padding every event left only about 14 days once the list had 13 events). Descriptive only: no test, no model.

In [ ]:
def spike_table(series):
    if events is None:
        return None
    pad, day = pd.Timedelta(days=1), pd.Timedelta(days=1)
    # Baseline (rule changed 23/11): drop every event's own window, and one extra day on each side
    # only for the major disruptions flagged exclude_from_baseline (Pope's visit, 17 Oct, the two closures).
    # Padding every event left only about 14 baseline days once the list had 13 events.
    in_ev = pd.Series(False, index=net.index)
    for r in events.itertuples():
        p_ = pad if r.exclude_from_baseline else pd.Timedelta(0)
        in_ev |= (net["ts"] >= r.date_start - p_) & (net["ts"] < r.date_end + p_ + day)
    base = net.loc[~in_ev, series]
    if series == "pct_oos":
        print(f"Baseline: {net.loc[~in_ev, 'ts'].dt.date.nunique()} days, {(~in_ev).sum():,} hours")
    if base.empty:
        print("No hour is more than one day away from every event window: baseline taken over all hours outside the windows themselves.")
        base = net.loc[~net["ts"].apply(lambda t: any(r.date_start <= t < r.date_end + day for r in events.itertuples())), series]
    rows = []
    for r in events.itertuples():
        w = net[(net["ts"] >= r.date_start) & (net["ts"] < r.date_end + day)]
        if w.empty:
            continue
        rows.append({"event": r.event_id, "kind": r.kind, "label": r.label, "peak_%": round(w[series].max(), 2),
                     "peak_at": w.loc[w[series].idxmax(), "ts"], "baseline_median_%": round(base.median(), 2),
                     "baseline_p95_%": round(base.quantile(0.95), 2),
                     "peak / baseline median": round(w[series].max() / base.median(), 1) if base.median() else None,
                     "hours above baseline p95 (%)": round(100 * (w[series] > base.quantile(0.95)).mean(), 0)})
    return pd.DataFrame(rows)

for s_ in ["pct_oos", "pct_cannot_rent"]:
    t = spike_table(s_)
    if t is not None:
        print(f"\n--- {s_} (hourly network share) ---")
        display(t)

In [ ]:
def say(*lines):
    print("\n".join(lines))

for s_, what in [("pct_oos", "out of service"), ("pct_cannot_rent", "cannot rent")]:
    t = spike_table(s_)
    if t is None:
        continue
    for r in t.itertuples():
        say(f"Observation ({what}), {r.event} {r.label}: peak {r._4}% at {r.peak_at}; baseline median {r._6}% and 95th percentile {r._7}%; "
            f"{r._9:.0f}% of the event's hours are above the baseline 95th percentile.")

**Station-level check (closures).** The operator's post lists the closed stations by station code. Copy each list into `data/external/closed_stations_<event_id>.csv` (one column, `stationCode`; note the access date in `events.csv`). The check then compares those stations with all the others inside the window: if the right stations went out of service (or empty) and the others did not, the measure sees what the operator did.

In [ ]:
def closed_station_check(event_id):
    f = EXT / f"closed_stations_{event_id}.csv"
    if events is None or not f.exists():
        print(f"{event_id}: no list yet (data/external/closed_stations_{event_id}.csv with a stationCode column).")
        return None
    r = events[events["event_id"] == event_id].iloc[0]
    codes = set(pd.read_csv(f, dtype=str)["stationCode"].str.strip())
    ps = panel_stations.assign(code=panel_stations["stationCode"].astype(str))
    listed = set(ps.loc[ps["code"].isin(codes), "station_id"])
    print(f"{event_id}: {len(codes)} codes in the list, {len(listed)} found among the {len(ps):,} panel stations")
    d = pd.to_datetime(sh_l["date_paris"])
    inside = (d >= r["date_start"]) & (d <= r["date_end"])
    out = []
    for name, mask in [("listed as closed", sh_l["station_id"].isin(listed)), ("all other stations", ~sh_l["station_id"].isin(listed))]:
        for when, w in [("inside window", inside), ("outside event windows", ~sh_l["date_paris"].isin(EVENT_DAYS))]:
            g = shares_alt(sh_l[mask & w].assign(all="x"), "all").iloc[0]
            out.append({"stations": name, "period": when, "% out of service": 100 * g["n_oos"] / g["n"],
                        "% cannot rent": g["pct_cannot_rent"], "% cannot return": g["pct_cannot_return"]})
    return pd.DataFrame(out).round(2)

for eid in (events["event_id"][events["kind"] == "planned_closure"] if events is not None else []):
    res = closed_station_check(eid)
    if res is not None:
        display(res)

**Count check (no list needed).** For each day and station we ask whether the station was out of service for at least half of the captures of that day, then count those stations per day. For a closure window we compare the count inside the window with the count on days outside every event window, and with the number the source announces (see `events.csv`). **Observation only:** this shows that a closure of about the announced size is visible; it does not show that the right stations closed. The same table lists the days **outside** any event window with the most stations out of service: a rise without an event is listed, not explained . The table is saved as `outputs/oos_by_day.csv`, the file the report quotes from.

In [ ]:
def oos_by_day():
    g = sh_l.groupby(["date_paris", "station_id"]).agg(oos=("n_oos", "sum"), n=("n", "sum")).reset_index()
    g["mostly"] = g["oos"] >= 0.5 * g["n"]
    t = g.groupby("date_paris").agg(oos=("oos", "sum"), n=("n", "sum"), stations_mostly_oos=("mostly", "sum"))
    t["pct_oos"] = (100 * t["oos"] / t["n"]).round(2)
    t.index = pd.to_datetime(t.index)
    t["weekday"] = t.index.day_name()
    t["event"] = ""
    if events is not None:
        for r in events.itertuples():
            m = (t.index >= r.date_start) & (t.index <= r.date_end)
            t.loc[m, "event"] = (t.loc[m, "event"] + " " + r.event_id).str.strip()
    return t[["pct_oos", "stations_mostly_oos", "weekday", "event"]]

if events is None:
    print("TODO: events.csv missing (section 0.8).")
else:
    oos_days = oos_by_day()
    oos_days.to_csv(OUT / "oos_by_day.csv")
    outside = oos_days.loc[oos_days["event"] == "", "stations_mostly_oos"]
    for r in events[events["kind"] == "planned_closure"].itertuples():
        inside = oos_days.loc[(oos_days.index >= r.date_start) & (oos_days.index <= r.date_end), "stations_mostly_oos"]
        print(f"Observation, {r.event_id} {r.label}: stations out of service at least half of the day: median {inside.median():.0f} and maximum {inside.max():.0f} "
              f"inside the window ({len(inside)} days), against a median of {outside.median():.0f} and a 95th percentile of {outside.quantile(0.95):.0f} on the {len(outside)} days outside every event window. "
              f"Compare the maximum with the number announced in the source.")
    print("\nDays outside every event window with the most stations out of service (listed, not explained):")
    display(oos_days[oos_days["event"] == ""].nlargest(12, "stations_mostly_oos").sort_index())

**Reading of section 6** (written 23.11, after checking the event sources).

**What we see:**

The two closures (E09 and E10) are the clearest signal. The 110 stations that Vélib' closed were out of service 72% of the time during E09 and 49% during E10, while outside the events they are only out of service 3.7% of the time. The other stations barely moved, going from 1.3% to 2.7% (E09) and 2.0% (E10). At network level, these two closures are also the biggest peaks of the period: in one hour, 11.1% and 9.7% of the stations were out of service, when the normal level is around 1.26%.

The peak of E08 is at 23:00 UTC on 22 Oct, which is 01:00 in Paris on 23 Oct. So it is the start of the E09 closure, and not the rally of the 22nd.

On march and strike days (E02, E05, E06, E07, E11, E12 and E13), the share of stations out of service is above the normal range (2.0%) for 88 to 100% of the hours. The peaks stay small (2.2 to 3.3%), but the daily value is about twice the one of the days around, for example 2.2% on 17 Oct against 1.0% on 16 and 18 Oct. We see no rise on 2 Oct (blockades but no march in Paris) or on 6 Oct. For "cannot rent", only the two closures really stand out.

**What we think:**

The measure works, because it finds the right stations at the right time while the other stations stay quiet. The small rise on march days is really in the data, but no source explains it. Maybe some stations were closed near the routes, but this is only a guess, so we just report it. The damage of 17 Oct (6 stations) does not show at network level, which is normal because it is too small. Also, the feed only gives stations and never bikes, so the numbers of bikes destroyed or stolen in the press are only context.

**How we did it:**

The normal level uses 29 days (650 hours). We kept all the hours outside the event windows, and we also removed one day around the biggest disruptions: the Pope's visit, 17 Oct and the two closures (flagged `exclude_from_baseline`).

We added E12 (5 Nov) and E13 (7 Nov) on 23.11 using the Interior Ministry's counts. We had already seen the results of the other events, but not the data of these two days. They were missing even though they are bigger in Paris than E11. Finally, the stations closed before 20.09 for the Pope's visit (Champs-Élysées sector) are not in our data.

## 7. Analysis

Progressive: from the whole network (how often), to the shape of the day (when), to the geography (where), to the question itself (no alternative within 300 m), then to how far we trust it (robustness). Everything reads the small tables of Part 0.

### 7.1 How often

Share of panel station-captures in which a station cannot rent / cannot take a return, the part with no alternative within 300 m, and the part that comes from out-of-service stations (the findings note reports about 2.3 points of each). **Observation** first; whether the numbers are "big or small" needs a yardstick: we have no external benchmark, so the yardstick is internal (by hour, day type, area, normal days against event days) and we say that the period is eight autumn weeks.

In [ ]:
tot = shares_alt(sh.assign(all="network"), "all").iloc[0]
how_often = pd.DataFrame({
    "cannot rent": [tot.pct_cannot_rent, tot.pct_cannot_rent_no_alt, 100 * tot.n_cannot_rent_oos / tot.n],
    "cannot take a return": [tot.pct_cannot_return, tot.pct_cannot_return_no_alt, 100 * tot.n_cannot_return_oos / tot.n],
}, index=["% of station-captures failing", "...of which no alternative within 300 m",
          "...of which out-of-service stations (any alternative)"]).round(2)
print(f"{int(tot.n):,} panel station-captures" + (" (sample days only)" if SAMPLE_DAYS else ""))
how_often

In [ ]:
say(f"Observation. A station cannot rent in {tot.pct_cannot_rent:.1f}% of panel station-captures and cannot take a return in {tot.pct_cannot_return:.1f}%.",
    f"With no other station within 300 m able to serve, the figures are {tot.pct_cannot_rent_no_alt:.1f}% (rent) and {tot.pct_cannot_return_no_alt:.1f}% (return): "
    f"{100 * tot.pct_cannot_rent_no_alt / tot.pct_cannot_rent:.0f}% and {100 * tot.pct_cannot_return_no_alt / tot.pct_cannot_return:.0f}% of the failing time.",
    f"Out-of-service stations account for {100 * tot.n_cannot_rent_oos / tot.n:.1f} points of the rent figure and {100 * tot.n_cannot_return_oos / tot.n:.1f} of the return figure.")

**Reading of 7.1** 

**What we can see:**

A station cannot rent in 6.9% of the captures and cannot take a return in 5.2%, which is about one capture in 14 for renting and one in 19 for returning. If we only count the failures where no other station within 300 m can serve, the figures fall to 2.2% (rent) and 2.0% (return). That is 31% and 38% of the failing time, so most of the time a station fails, another one nearby still works.

Out-of-service stations explain 2.35 points of each figure. That is about a third of the rent failures (34%) and almost half of the return failures (45%). The rest comes from stations that work but are empty or full. Over the whole period renting fails more often than returning, but the next section shows that it depends on the hour. Without the 14 event days, the figures are a bit lower (6.2% for rent and 4.4% for return, see 7.5).

**What we think:**

About 1 failure in 3 has no alternative nearby, and this share is a bit higher for returning than for renting. Section 7.4 shows that 71% (rent) and 74% (return) of this "no alternative" time come from isolated stations, and only the rest from neighbours that fail at the same moment.

We have no outside benchmark to say whether 6.9% is a lot or not, so we read it inside the data in the next sections (by hour, by area, with and without events). A failing station is not a person without a bike: we measure stations and not users, and someone can also walk further or wait. The period is only eight autumn weeks, so we cannot say anything about summer or winter.

### 7.2 When  ·  **report figure 2**

Hour of day (Paris time) by weekday. **Observation:** read the pattern off the heatmaps. **Interpretation** (to write after looking): commuting, the day of the week, the evening return.

In [ ]:
def heat(df, col_):
    return df.pivot(index="weekday_paris", columns="hour_paris", values=col_).reindex(range(7)).reindex(columns=range(24))

hw = shares_alt(sh_l, ["weekday_paris", "hour_paris"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, (c, title) in zip(axes, [("pct_cannot_rent", "cannot rent"), ("pct_cannot_return", "cannot take a return")]):
    m = heat(hw, c)
    im = ax.imshow(m.to_numpy(), aspect="auto", cmap="viridis")
    ax.set_yticks(range(7)); ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    ax.set_xticks(range(0, 24, 3)); ax.set_xlabel("hour of day (Paris time)"); ax.set_title(title)
    fig.colorbar(im, ax=ax, label="% of station-captures")
fig.tight_layout(); fig.savefig(OUT / "fig_hour_weekday.png", dpi=150); plt.show()

In [ ]:
wd = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
for c, title in [("pct_cannot_rent", "cannot rent"), ("pct_cannot_return", "cannot take a return")]:
    m = heat(hw, c)
    (d_, h_), top = m.stack().idxmax(), m.stack().max()
    say(f"Observation ({title}): highest cell {wd[int(d_)]} {int(h_)}:00 Paris time at {top:.1f}%; lowest {m.stack().min():.1f}%. "
        f"Mean of the hourly cells: weekdays {m.loc[0:4].mean().mean():.1f}%, weekend " + (f"{m.loc[5:6].mean().mean():.1f}%" if m.loc[5:6].notna().any().any() else "n/a (no weekend in the days loaded)") + ". "
        f"Busiest hour of the weekday average: {int(m.loc[0:4].mean().idxmax())}:00.")

In [ ]:
# the same heatmaps in numbers: average of the weekday cells and of the weekend cells, by hour of the day
for c, title in [("pct_cannot_rent", "cannot rent"), ("pct_cannot_return", "cannot take a return")]:
    m = heat(hw, c)
    wk, we = m.loc[0:4].mean(), m.loc[5:6].mean()
    say(f"Observation ({title}), weekdays: night (0:00 to 5:00) {wk.loc[0:5].mean():.1f}%, 9:00 to 13:00 {wk.loc[9:13].mean():.1f}%, 17:00 to 19:00 {wk.loc[17:19].mean():.1f}%; "
        f"lowest hour {int(wk.idxmin())}:00 ({wk.min():.1f}%), highest hour {int(wk.idxmax())}:00 ({wk.max():.1f}%).",
        f"Observation ({title}), weekend: night {we.loc[0:5].mean():.1f}%, 15:00 to 17:00 {we.loc[15:17].mean():.1f}%; "
        f"lowest hour {int(we.idxmin())}:00 ({we.min():.1f}%), highest hour {int(we.idxmax())}:00 ({we.max():.1f}%).",
        f"Observation ({title}), by day (mean of the hourly cells): " + ", ".join(f"{d} {v:.1f}%" for d, v in zip(wd, m.mean(axis=1))) + ".")

**Reading of 7.2**

**What we observe:**

On weekdays, cannot rent is around 5% at night, rises quickly from 8:00 and stays close to 10% between 9:00 and 13:00 (10.0% at 11:00). It then goes back down during the afternoon to about 5% in the evening. Cannot take a return behaves differently: it is lowest at 8:00 (2.9%) and during the evening rush (3.4% on average between 17:00 and 19:00), and highest at night (6.1% at 2:00) and at 10:00. So on weekdays the two sides do not peak at the same time: renting is the hardest in the middle of the day and returning at night.

At the weekend the pattern is flatter. Cannot rent peaks in the afternoon (10.1% at 16:00) and cannot take a return peaks at the end of the night (7.6% at 4:00). Saturday is the worst day (8.1% for rent and 6.8% for return, against 6.0 to 7.3% and 4.3 to 5.3% on the weekdays). The highest cell of the heatmap is Saturday 16:00 for renting (11.4%) and Saturday 4:00 for returning (8.0%).

**What we think :**

The weekday pattern for renting fits with commuting: bikes leave in the morning and they are not all back at midday. It is only a possible reading, because we do not follow the bikes and we have no flows per area. The night level (5 to 7%) is not a good sign: at night nothing moves, so the figure stays stable, and part of it is made of stations that are out of service (2.35 points over the whole period). For the Saturday afternoon, leisure trips are possible but we did not test it. We also cannot explain why returning is easiest during the two rush hours.

The heatmaps include the event days, which raise the average a bit (look at section 7.5), so we use them for the general shape and not for exact values.

### 7.3 Where  ·  **report figure 3**

Share of time each station cannot rent / cannot take a return, on a plain map (longitude and latitude, no basemap). **Observation:** where the dark dots are. Beware of reading a station's neighbourhood into a single dot: a commune is an administrative unit, not a catchment area.

In [ ]:
st = shares_alt(sh, "station_id").merge(panel_stations[["station_id", "stationCode", "name", "lat", "lon"]], on="station_id")
asp = 1 / np.cos(np.radians(st["lat"].mean()))
fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
for ax, (c, title) in zip(axes, [("pct_cannot_rent", "cannot rent"), ("pct_cannot_return", "cannot take a return")]):
    o = st.sort_values(c)
    sc = ax.scatter(o["lon"], o["lat"], c=o[c], s=8, cmap="magma_r", vmax=o[c].quantile(0.98))
    ax.set_aspect(asp); ax.set_title(f"Share of time a station {title}"); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(sc, ax=ax, shrink=0.8, label="% of captures")
fig.tight_layout(); fig.savefig(OUT / "fig_station_map.png", dpi=150); plt.show()

if communes is None:
    print("TODO: Paris against the rest needs data/external/station_communes.csv (section 0.4).")
else:
    by_paris = shares_alt(sh.merge(communes[["station_id", "is_paris"]], on="station_id", how="left"), "is_paris")
    display(by_paris[["is_paris", "n", "pct_cannot_rent", "pct_cannot_rent_no_alt", "pct_cannot_return", "pct_cannot_return_no_alt"]].round(2))

In [ ]:
for c, title in [("pct_cannot_rent", "cannot rent"), ("pct_cannot_return", "cannot take a return")]:
    o = st.sort_values(c, ascending=False)
    say(f"Observation ({title}): median station {st[c].median():.1f}%, 90th percentile {st[c].quantile(.9):.1f}%, maximum {st[c].max():.1f}%. "
        f"{int((st[c] > 20).sum())} of {len(st):,} stations are above 20% of the time. Worst five: " + "; ".join(f"{r.name} ({getattr(r, c):.0f}%)" for r in o.head(5).itertuples()))
if communes is not None:
    for r in by_paris.itertuples():
        say(f"Observation, {'Paris' if r.is_paris else 'rest of the network'}: cannot rent {r.pct_cannot_rent:.1f}% (no alternative {r.pct_cannot_rent_no_alt:.1f}%), "
            f"cannot take a return {r.pct_cannot_return:.1f}% (no alternative {r.pct_cannot_return_no_alt:.1f}%).")

In [ ]:
# how concentrated are the failures, and how far from the centre (rough zones, straight line to Notre-Dame)
for c, f, fo, title in [("pct_cannot_rent", "n_cannot_rent", "n_cannot_rent_oos", "cannot rent"),
                        ("pct_cannot_return", "n_cannot_return", "n_cannot_return_oos", "cannot take a return")]:
    w = st[st[c] > 20]
    say(f"Observation ({title}): the {len(w)} stations above 20% ({100 * len(w) / len(st):.1f}% of the stations) produce {100 * w[f].sum() / st[f].sum():.0f}% of all the failing station-captures; "
        f"out-of-service time is {100 * w[fo].sum() / w[f].sum():.0f}% of their failing time.")
say(f"Stations out of service during the whole period: {int((st['n_oos'] == st['n']).sum())}.")

x = (st["lon"] - 2.3488) * np.cos(np.radians(48.85)) * 111.32
y = (st["lat"] - 48.8534) * 110.57
st["km_centre"] = np.hypot(x, y)
st["zone"] = pd.cut(st["km_centre"], [0, 2, 4, 6, 10, 100], labels=["0-2 km", "2-4 km", "4-6 km", "6-10 km", "over 10 km"])
zones = st.groupby("zone", observed=True).agg(stations=("station_id", "count"),
        rent_mean=("pct_cannot_rent", "mean"), rent_median=("pct_cannot_rent", "median"),
        return_mean=("pct_cannot_return", "mean"), return_median=("pct_cannot_return", "median")).round(1)
display(zones)

if communes is not None:
    cm = st.merge(communes[["station_id", "commune", "is_paris"]], on="station_id")
    cm = cm[~cm["is_paris"]].groupby("commune").agg(stations=("station_id", "count"),
            rent_median=("pct_cannot_rent", "median"), return_median=("pct_cannot_return", "median")).query("stations >= 15")
    print("Communes outside Paris with at least 15 stations, highest median 'cannot take a return':")
    print(cm.sort_values("return_median", ascending=False).round(1).head(4).to_string())

**Reading of 7.3**

**What we see:**

The failures are concentrated in a small group of stations. The median station cannot rent 3.9% of the time and cannot take a return 2.0%, but the 90th percentile is at 15.8% and 13.2%. The 91 stations that cannot rent more than 20% of the time (6% of the network) produce 31% of all the rent failures. For returning, 55 stations (3.6%) produce 29% of the failures. Out-of-service time is most of their failing time (65% for the 91 rent stations and 86% for the 55 return stations), and 8 stations were out of service during the whole period, for example Réaumur - Montmartre and Marignan - Champs-Elysées. This is why the worst stations are at 100%.

Most of these stations are in Paris (81 of the 91 for renting). The median station in Paris cannot rent 6.0% of the time, against 1.2% outside Paris, and cannot take a return 2.2% against 1.8%. Outside Paris the share of failing captures (4.3% for rent) is much higher than the median station (1.2%), so a few stations with very long failures weigh a lot in the average.

The distance to the centre of Paris (straight line to Notre-Dame, only a rough zone) shows the same thing. The 197 stations within 2 km cannot rent 9.4% of the time and cannot take a return 11.8% on average, against 3.8% and 4.8% for the 354 stations between 6 and 10 km. The return side is the most central: from 2 to 4 km the average falls to 4.9%. Outside Paris, the highest median for cannot take a return is in Ivry-sur-Seine (7.0%), Issy-les-Moulineaux and Boulogne-Billancourt (6.1%), where the median for cannot rent is below 0.5%.

**What we think:**

The failures are not spread evenly. A small group of stations, mostly in Paris and often out of service for long periods, makes up about a third of the failing time. So the average hides two situations: stations that work but are often empty or full, and stations that are simply closed for weeks.

The centre is the hot spot, especially for returning, but we do not know if it comes from the demand, the terrain or the maintenance, so we only report it. In Ivry, Issy and Boulogne the stations are often full but almost never empty (if they were out of service, renting would fail too). We do not try to explain it. A dot on the map is one station and not a neighbourhood, and a commune is an administrative unit and not a catchment area. Finally, Paris and the rest do not have the same share of isolated stations (see 7.4), so a comparison of "no alternative" between them mixes two things.

### 7.4 The 300 m question  ·  **report table, Paris against the rest**

Failures with no alternative within 300 m, for isolated stations (no neighbour within 300 m, no alternative by construction) and for the others. **Observation:** how much of the failing time is "no alternative", and how much of it is simply isolation. The same with 200 m and 500 m is done at the end of this section (radius test). The aim of the whole project is to tell a user where an alternative within 300 m exists *and works*.

In [ ]:
iso = set(panel_stations["station_id"]) - set(station_pairs_300m["station_id"])
t = shares_alt(sh.assign(isolated=sh["station_id"].isin(iso).map({True: "isolated", False: "has a neighbour within 300 m"})), "isolated")
t["stations"] = t["isolated"].map(sh.assign(isolated=sh["station_id"].isin(iso).map({True: "isolated", False: "has a neighbour within 300 m"}))
                                   .groupby("isolated")["station_id"].nunique())
t["rent: % of failures with no alternative"] = 100 * (t["n_cannot_rent"] - t["n_cannot_rent_alt"]) / t["n_cannot_rent"]
t["return: % of failures with no alternative"] = 100 * (t["n_cannot_return"] - t["n_cannot_return_alt"]) / t["n_cannot_return"]
cols_ = ["isolated", "stations", "pct_cannot_rent", "pct_cannot_rent_no_alt", "rent: % of failures with no alternative",
         "pct_cannot_return", "pct_cannot_return_no_alt", "return: % of failures with no alternative"]
t[cols_].round(2)

In [ ]:
for r in t.itertuples():
    say(f"Observation, {r.isolated} ({int(r.stations)} stations): cannot rent {r.pct_cannot_rent:.1f}% of the time, of which no alternative {r.pct_cannot_rent_no_alt:.1f}%; "
        f"cannot take a return {r.pct_cannot_return:.1f}%, of which no alternative {r.pct_cannot_return_no_alt:.1f}%.")

**Paris against the rest, split by isolated and not isolated.** In 7.3 the rest of the network has more failures with no alternative than Paris (3.7% against 1.4% for renting), even though its stations fail less often. Before reading this as "the suburbs are worse served", we check if it simply comes from the fact that stations outside Paris are more often alone in their 300 m radius. So we cross the two splits: area (Paris or not) and isolated (no other panel station within 300 m) or not.

In [ ]:
if communes is None:
    print("TODO: needs data/external/station_communes.csv (section 0.4).")
else:
    pi = sh.merge(communes[["station_id", "is_paris"]], on="station_id", how="left")
    pi["area"] = pi["is_paris"].map({True: "Paris", False: "rest of the network"})
    pi["isolated"] = pi["station_id"].isin(iso).map({True: "isolated", False: "has a neighbour"})
    cross = shares_alt(pi, ["area", "isolated"])
    cross = cross.merge(pi.groupby(["area", "isolated"], as_index=False)["station_id"].nunique()
                          .rename(columns={"station_id": "stations"}), on=["area", "isolated"])
    cross["rent: % of failures with no alternative"] = 100 * (cross["n_cannot_rent"] - cross["n_cannot_rent_alt"]) / cross["n_cannot_rent"]
    cross["return: % of failures with no alternative"] = 100 * (cross["n_cannot_return"] - cross["n_cannot_return_alt"]) / cross["n_cannot_return"]
    display(cross[["area", "isolated", "stations", "pct_cannot_rent", "pct_cannot_rent_no_alt", "rent: % of failures with no alternative",
                   "pct_cannot_return", "pct_cannot_return_no_alt", "return: % of failures with no alternative"]].round(2))

In [ ]:
if communes is not None:
    # 1. how many stations are isolated in each area
    for a, g in cross.groupby("area"):
        n_iso = int(g.loc[g["isolated"] == "isolated", "stations"].sum())
        say(f"Observation, {a}: {n_iso} of {int(g['stations'].sum())} stations are isolated ({100 * n_iso / g['stations'].sum():.1f}%).")

    # 2. how many neighbours the non-isolated stations have
    nn = station_pairs_300m.groupby("station_id").size().rename("neighbours").reset_index().merge(communes[["station_id", "is_paris"]], on="station_id")
    for k, g in nn.groupby("is_paris"):
        say(f"Observation, {'Paris' if k else 'rest of the network'}: a station with a neighbour has a median of {g['neighbours'].median():.0f} neighbours "
            f"within 300 m, and {100 * (g['neighbours'] == 1).mean():.0f}% of them have only one.")

    # 3. what if the rest of the network had as many isolated stations as Paris? (reweighting, not a model)
    w = cross[cross["area"] == "Paris"].set_index("isolated")["n"]
    w = w / w.sum()
    rest = cross[cross["area"] == "rest of the network"].set_index("isolated")
    paris = cross[cross["area"] == "Paris"].set_index("isolated")
    for c, side in [("pct_cannot_rent_no_alt", "cannot rent"), ("pct_cannot_return_no_alt", "cannot take a return")]:
        real_rest = (rest[c] * rest["n"]).sum() / rest["n"].sum()
        real_paris = (paris[c] * paris["n"]).sum() / paris["n"].sum()
        say(f"Check ({side}, no alternative): rest of the network {real_rest:.2f}%, Paris {real_paris:.2f}%. "
            f"With the isolated share of Paris ({100 * w['isolated']:.1f}% of captures), the rest of the network would be at {(rest[c] * w).sum():.2f}%.")

**Radius test: 200 m and 500 m.** The 300 m comes from the spacing that Vélib' Métropole states for Paris (see section 1), but it is still a choice, and it does not hold for the suburbs. So we redo the whole "alternative" computation (0.5 and 0.7) with 200 m and 500 m, nothing else changes. The first run builds two tables in `data/derived/` (it takes a few minutes per radius, like 0.7), after that they are read from the cache. If the answer barely moves, the 300 m choice does not drive the result; if it moves, we say by how much.

In [ ]:
def pairs_for(radius_m):
    # same formula and same steps as build_pairs_300m (0.5), only the radius changes
    allp = pairs_within(pd.read_csv(NEIGH), radius_m)
    ok = set(panel_stations["station_id"])
    return allp[allp["station_id"].isin(ok) & allp["neighbour_id"].isin(ok)].reset_index(drop=True)

assert len(pairs_for(300)) == len(station_pairs_300m), "the 300 m pairs rebuilt here should equal the table of 0.5"

radius_rows = []
for r_m in (200, 300, 500):
    p_ = station_pairs_300m if r_m == 300 else pairs_for(r_m)
    alt_ = station_hour_alt if r_m == 300 else cached(f"station_hour_alt_{r_m}m", lambda: build_station_hour_alt(p_))
    d_ = station_hour.merge(alt_, on=["station_id", "date_utc", "hour_utc"], how="left")
    d_[ALT] = d_[ALT].fillna(0).astype(int)
    has = set(p_["station_id"])
    row = {"radius (m)": r_m, "% of panel stations with a neighbour": 100 * len(has) / len(panel_stations)}
    tot_ = shares_alt(d_.assign(all="x"), "all").iloc[0]
    row["rent: % no alternative"] = tot_.pct_cannot_rent_no_alt
    row["rent: % of failures with no alternative"] = 100 * tot_.pct_cannot_rent_no_alt / tot_.pct_cannot_rent
    row["return: % no alternative"] = tot_.pct_cannot_return_no_alt
    row["return: % of failures with no alternative"] = 100 * tot_.pct_cannot_return_no_alt / tot_.pct_cannot_return
    if communes is not None:
        d_ = d_.merge(communes[["station_id", "is_paris"]], on="station_id", how="left")
        for k, g in shares_alt(d_, "is_paris").set_index("is_paris").iterrows():
            a = "Paris" if k else "rest"
            ids = set(communes.loc[communes["is_paris"] == k, "station_id"]) & set(panel_stations["station_id"])
            row[f"{a}: % stations isolated"] = 100 * len(ids - has) / len(ids)
            row[f"{a}: rent % no alternative"] = g["pct_cannot_rent_no_alt"]
            row[f"{a}: return % no alternative"] = g["pct_cannot_return_no_alt"]
    radius_rows.append(row)
con.register("pairs", station_pairs_300m[["station_id", "neighbour_id"]])   # back to the 300 m table
radius_test = pd.DataFrame(radius_rows).set_index("radius (m)").round(2)
radius_test.T

In [ ]:
for r_m, r in radius_test.iterrows():
    say(f"Observation, {r_m} m: {r['% of panel stations with a neighbour']:.0f}% of stations have a neighbour; no alternative "
        f"{r['rent: % no alternative']:.2f}% (rent, {r['rent: % of failures with no alternative']:.0f}% of failures) and "
        f"{r['return: % no alternative']:.2f}% (return, {r['return: % of failures with no alternative']:.0f}% of failures)"
        + (f"; Paris {r['Paris: rent % no alternative']:.2f}% against rest {r['rest: rent % no alternative']:.2f}% (rent)." if "Paris: rent % no alternative" in r else "."))

**Reading of 7.4**

**What we can see:**

Stations that have a neighbour fail more often than isolated ones (7.9% against 4.9% for renting), but most of the time another station within 300 m can still help. Only 11.5% of their rent failures and 13.5% of their return failures have no alternative. For isolated stations it is 100% by definition, so most of the "no alternative" time comes from them.

When we cross area and isolation, the gap between Paris and the rest becomes clearer. Only 8.9% of the stations in Paris are isolated, against 76.1% outside Paris. Outside Paris, even the stations with a neighbour usually have only one (3 out of 4 of them), while in Paris the median is 3. If the rest of the network had the same share of isolated stations as Paris, its "no alternative" level would go from 3.7% to 0.9% for renting (Paris: 1.4%), and from 3.8% to 1.0% for returning (Paris: 1.0%). So with the same mix of isolated stations, the rest of the network would be at the level of Paris or a bit below, which means that over the whole period isolation is enough to explain the gap.

The radius changes the numbers a lot. With 200 m, 61% of rent failures have no alternative, with 300 m it is 31%, and with 500 m only 9%. Paris and the rest are almost equal at 200 m (4.3% and 4.1% for renting), but at 300 m and 500 m the rest of the network is clearly behind (3.7% against 1.4%, then 1.6% against 0.1%).

**What we think:**

The difference between Paris and the rest does not come from stations failing more often outside Paris, because they actually fail less. It comes from the network being less dense there, so when a station fails there is usually no other one nearby. Outside Paris, the problem is not that stations fail a lot, it's that a failing station is usually alone.

The 300 m choice matters a lot for the level of the result, so we always give the number together with its radius. What stays the same at 300 m and 500 m is the direction: the rest of the network is worse off than Paris, and the gap mostly comes from isolation. At 200 m almost half of the Paris stations (44%) are isolated too, which is consistent with the spacing of about 300 m that Vélib' Métropole states for Paris.

**Limits:**

The distance is a straight line between two stations, so a real walk is longer and our alternatives are a bit optimistic. The reweighting is only a check and not a model: it just says what the number would be with the same mix of isolated stations as in Paris.

### 7.5 Robustness: how far do we trust it?

The same headline table, (a) on all days, (b) without the days flagged `exclude_from_baseline` in `events.csv` (closures, the main damage day, the Pope's visit), and (c) a few extra checks (which events drive the difference, Paris against the rest, first and second half of the period, without the stations closed all the time). VM2 is discussed in the reading: the 300 m analysis was not rerun on it. **Interpretation rule:** if the numbers barely move without the event days, the result is not an artefact of the unusual weeks; if they move, say by how much.

In [ ]:
if not EVENT_DAYS:
    print("TODO: needs events.csv with exclude_from_baseline days (section 0.8).")
else:
    keep = ~sh_l["date_paris"].isin(EVENT_DAYS)
    cmp = pd.concat({"all days": shares_alt(sh_l.assign(all="x"), "all").iloc[0],
                     "without event days": shares_alt(sh_l[keep].assign(all="x"), "all").iloc[0]}, axis=1)
    print(f"{len(EVENT_DAYS)} days excluded ({100 * (~keep).mean():.1f}% of station-hours)")
    cmp.to_csv(OUT / f"headline_{VM}{'_sample' if SAMPLE_DAYS else ''}.csv")
    display(cmp.loc[["n", "pct_cannot_rent", "pct_cannot_rent_no_alt", "pct_cannot_return", "pct_cannot_return_no_alt"]].round(2))

In [ ]:
if EVENT_DAYS:
    for k, lab in [("pct_cannot_rent", "cannot rent"), ("pct_cannot_rent_no_alt", "cannot rent, no alternative"),
                   ("pct_cannot_return", "cannot take a return"), ("pct_cannot_return_no_alt", "cannot take a return, no alternative")]:
        a, b = cmp.loc[k, "all days"], cmp.loc[k, "without event days"]
        say(f"Observation, {lab}: {a:.2f}% on all days, {b:.2f}% without event days (difference {a - b:+.2f} points).")

In [ ]:
# more robustness checks, all on the same small tables (no new data)
ev_of = {}
for r in events[events["exclude_from_baseline"]].itertuples():
    for x in pd.date_range(r.date_start, r.date_end).date:
        ev_of[x] = r.event_id
rb = sh_l.copy()
rb["event"] = rb["date_paris"].map(ev_of)
rb["isolated"] = rb["station_id"].isin(iso)
normal = rb["event"].isna()

def no_alt_share(df):
    g = shares_alt(df.assign(all="x"), "all").iloc[0]
    return 100 * g.pct_cannot_rent_no_alt / g.pct_cannot_rent, 100 * g.pct_cannot_return_no_alt / g.pct_cannot_return

# 1. share of the failing time with no alternative, with and without event days
for lab, m in [("all days", rb.index == rb.index), ("without event days", normal)]:
    a, b = no_alt_share(rb[m])
    say(f"Observation, {lab}: {a:.1f}% of rent failures and {b:.1f}% of return failures have no alternative.")

# 2. day-to-day spread on normal full days (at least 23 hours of data), to compare with the differences above
dd = shares_alt(rb[normal], "date_paris")
hours = rb[normal].groupby("date_paris")["hour_paris"].nunique()
dd = dd[dd["date_paris"].map(hours) >= 23]
say(f"Observation: on the {len(dd)} full days without events, cannot rent goes from {dd['pct_cannot_rent'].min():.1f}% to {dd['pct_cannot_rent'].max():.1f}% "
    f"(median {dd['pct_cannot_rent'].median():.1f}%) and cannot take a return from {dd['pct_cannot_return'].min():.1f}% to {dd['pct_cannot_return'].max():.1f}% "
    f"(median {dd['pct_cannot_return'].median():.1f}%).")

# 3. which events drive the difference: remove one event at a time
tot_ = shares_alt(rb.assign(all="x"), "all").iloc[0]
for e in sorted(rb["event"].dropna().unique()):
    g = shares_alt(rb[rb["event"].fillna("") != e].assign(all="x"), "all").iloc[0]
    say(f"Observation, removing only {e}: cannot rent changes by {g.pct_cannot_rent - tot_.pct_cannot_rent:+.2f} points, "
        f"cannot take a return by {g.pct_cannot_return - tot_.pct_cannot_return:+.2f} points.")

# 4. Paris against the rest without event days, and the same reweighting as in 7.4
if communes is not None:
    rbn = rb[normal].merge(communes[["station_id", "is_paris"]], on="station_id", how="left")
    ar = shares_alt(rbn, "is_paris").set_index("is_paris")
    for k, r in ar.iterrows():
        say(f"Observation, without event days, {'Paris' if k else 'rest of the network'}: cannot rent {r.pct_cannot_rent:.2f}% (no alternative {r.pct_cannot_rent_no_alt:.2f}%), "
            f"cannot take a return {r.pct_cannot_return:.2f}% (no alternative {r.pct_cannot_return_no_alt:.2f}%).")
    cr = shares_alt(rbn, ["is_paris", "isolated"]).set_index(["is_paris", "isolated"])
    w_ = cr.loc[True, "n"] / cr.loc[True, "n"].sum()
    for c, side in [("pct_cannot_rent_no_alt", "cannot rent"), ("pct_cannot_return_no_alt", "cannot take a return")]:
        say(f"Check, without event days ({side}, no alternative): with the isolated share of Paris the rest of the network would be at {(cr.loc[False, c] * w_).sum():.2f}%.")
    na_r = rb["n_cannot_rent"] - rb["n_cannot_rent_alt"]
    na_e = rb["n_cannot_return"] - rb["n_cannot_return_alt"]
    for lab, m in [("all days", rb.index == rb.index), ("without event days", normal)]:
        say(f"Observation, {lab}: isolated stations make {100 * na_r[m & rb['isolated']].sum() / na_r[m].sum():.0f}% of the rent time with no alternative "
            f"and {100 * na_e[m & rb['isolated']].sum() / na_e[m].sum():.0f}% of the return time with no alternative.")

# 5. first half against second half of the period (normal days only)
days_ = sorted(rb["date_paris"].unique())
mid = days_[len(days_) // 2]
for lab, m in [(f"first half (before {mid})", rb["date_paris"] < mid), (f"second half (from {mid})", rb["date_paris"] >= mid)]:
    g = shares_alt(rb[m & normal].assign(all="x"), "all").iloc[0]
    say(f"Observation, {lab}, normal days: cannot rent {g.pct_cannot_rent:.2f}% (no alternative {g.pct_cannot_rent_no_alt:.2f}%), "
        f"cannot take a return {g.pct_cannot_return:.2f}% (no alternative {g.pct_cannot_return_no_alt:.2f}%).")

# 6. without the stations that were out of service during the whole period
per_st = sh.groupby("station_id")[["n", "n_oos"]].sum()
always_closed = set(per_st.index[per_st["n_oos"] == per_st["n"]])
g = shares_alt(rb[~rb["station_id"].isin(always_closed)].assign(all="x"), "all").iloc[0]
a, b = no_alt_share(rb[~rb["station_id"].isin(always_closed)])
say(f"Observation, without the {len(always_closed)} stations out of service all the time: cannot rent {g.pct_cannot_rent:.2f}%, cannot take a return {g.pct_cannot_return:.2f}%; "
    f"{a:.1f}% and {b:.1f}% of the failures have no alternative.")


**VM2 comparison (reliability check, never merged).** The second collector recorded the same network independently. If it gives the same picture as VM1, the result does not depend on one collector. How to run it is in the comment at the top of the next cell.

In [ ]:
# VM2 (reliability check, never merged). To produce it: set VM = "vm2" at the top, Run All (the self-check only prints, it does not stop),
# then set VM back to "vm1" and run this cell. Each run saves outputs/headline_<vm>.csv in section 7.5.
f1, f2 = OUT / "headline_vm1.csv", OUT / "headline_vm2.csv"
if f1.exists() and f2.exists():
    a, b = [pd.read_csv(f, index_col=0).apply(pd.to_numeric, errors="coerce") for f in (f1, f2)]
    vm_cmp = pd.concat({"VM1 all days": a["all days"], "VM2 all days": b["all days"],
                        "VM1 without event days": a["without event days"], "VM2 without event days": b["without event days"]}, axis=1)
    display(vm_cmp.loc[["n", "pct_cannot_rent", "pct_cannot_rent_no_alt", "pct_cannot_return", "pct_cannot_return_no_alt"]].round(2))
    for k, lab in [("pct_cannot_rent", "cannot rent"), ("pct_cannot_return", "cannot take a return")]:
        say(f"Observation, {lab}: VM1 {vm_cmp.loc[k, 'VM1 all days']:.2f}% against VM2 {vm_cmp.loc[k, 'VM2 all days']:.2f}% on all days.")
else:
    print("VM2 comparison not available yet: needs outputs/headline_vm1.csv and outputs/headline_vm2.csv (see the comment above).")

**Reading of 7.5**

**What we see:**

Without the 14 days flagged `exclude_from_baseline` (Pope's visit, 17 Oct and the two closures), which are 24.6% of the station-hours, cannot rent goes from 6.94% to 6.16% and cannot take a return from 5.21% to 4.37%. The "no alternative" part goes from 2.17% to 1.85% for renting and from 1.96% to 1.55% for returning. The share of failures with no alternative almost does not move: 30% instead of 31% for renting, and 35% instead of 38% for returning.

These differences (about 0.8 points) are small compared with how much the figure changes from one normal day to another. On the 43 full days without events, cannot rent goes from 3.9% to 9.6% and cannot take a return from 2.6% to 7.0%. About half of the difference comes from the E09 closure alone (0.39 of the 0.78 points for renting), then from E10 (0.18). The Pope's visit only adds 0.07 points and 17 Oct nothing.

Without the event days, Paris changes much more than the rest. In Paris cannot rent goes from 8.3% to 7.1%, while outside Paris it stays around 4.3%. For returning, Paris (4.38%) and the rest (4.35%) even become equal. The "no alternative" gap is still there: 0.9% in Paris against 3.7% outside for renting, and 0.4% against 3.7% for returning. With the share of isolated stations of Paris, the rest of the network would be at 0.9% for renting and 1.0% for returning. Isolated stations now make 81% and 86% of the time with no alternative (71% and 74% on all days).

2 more checks! The 1st and the 2nd half of the period (normal days only) give almost the same figures for renting (6.3% then 6.0%, no alternative 1.9% then 1.8%), but cannot take a return with no alternative is lower in the second half (1.8% then 1.3%). Without the 8 stations that were out of service all the time, the headline goes down to 6.4% and 4.7%, and the share of failures with no alternative stays at 30% and 37%.

**What we think:**

The main results do not come from the unusual weeks. Without them failures are a bit rarer, but about one failure in three still has no alternative, and the gap between Paris and the rest still comes mostly from isolation. For renting, isolation explains almost all of the gap (0.9% against 0.9% in Paris). For returning it explains most of it but not all (3.7% goes down to 1.0%, while Paris is at 0.4%).

The events mostly hit Paris, because the closed stations were there. They also created "no alternative" time at stations that normally have neighbours, since when a whole area is closed the neighbours are closed too. This is why the share of isolated stations in the "no alternative" time goes up once the event days are removed.

One thing is less stable: returning with no alternative went down between the two halves of the period. We cannot say why (weather, season or maintenance are all possible), so we only report it.

**What we could not check:**

VM2 was compared with VM1 in the cleaning pipeline: the 2 collectors give the same shares of time failing within 0.1 point and almost the same number of episodes (124,950 against 125,031 for cannot rent). We also reran the whole 300 m analysis on VM2 (section 7.5, `VM = "vm2"`) and the "no alternative" figures are the same as on VM1: 2.17% against 2.17% for renting and 1.96% against 1.96% for returning (1.85% against 1.85% and 1.55% against 1.55% without the event days). The biggest gap on any headline figure is 0.004 points, so the result does not depend on which VM we use.

## 8. Reproducibility: anyone can redo it

Reproducibility is graded, and it was a deliberate choice: three layers, each rebuilt from public pieces.

| To redo... | You need | Where |
|---|---|---|
| **The collection** | the VM set-up guide and the cron code | GitHub: `vm_setup/`, `code/` |
| **The cleaning** (raw to clean) | the raw tars, then `python code/run_all.py --vm vm1 --no-bench` (about 2 hours) | Zenodo (DOI 10.5281/zenodo.24444738) and GitHub |
| **The analysis** | the cleaned Parquet and this notebook | Zenodo and GitHub: `notebooks/` |

Other students already reran the scripts from the raw files during the peer reviews. **What cannot be reproduced exactly:** the live feed keeps no history, so the same 57 days cannot be collected again; the raw archive on Zenodo takes its place. **Tested on another machine?** Yes: the notebook gave identical figures on two Windows laptops, and the self-check (Part 0.2) passed on both.

**Environment.** The next cell records the machine and the package versions that produced the numbers, in `outputs/environment.txt`, so that the reported figures can be traced to a setup.

In [ ]:
# Record the environment that produced the reported numbers (reproducibility): commit outputs/environment.txt with the report.
import importlib.metadata as md_, datetime as dt_
lines = [f"date: {dt_.datetime.now().isoformat(timespec='seconds')}", f"sample_days: {SAMPLE_DAYS}", f"machine: {_machine() if '_machine' in globals() else platform.platform()}"]
for pkg in ["duckdb", "pandas", "pyarrow", "numpy", "matplotlib", "requests", "imageio-ffmpeg"]:
    try:
        lines.append(f"{pkg}: {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        lines.append(f"{pkg}: not installed")
(OUT / "environment.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## 9. Conclusion, limits and lessons learned

**The answer, in 3 sentences.**
1. *How often.* Over the 8 weeks, a station cannot rent 6.9% of the time and cannot take a return 5.2% of the time, and in 2.2% and 2.0% of the time no other station within 300 m can serve instead (31% and 38% of the failing time).
2. *Why.* Most of this "no alternative" time comes from the 487 isolated stations (71% for renting, 74% for returning). Stations with a neighbour are rarely left without an alternative: only 11.5% of their rent failures and 13.5% of their return failures.
3. *Paris and the rest.* Stations in Paris fail more often (8.3% and 5.6% against 4.3% and 4.5%), but outside Paris a failure much more often has no alternative (3.7% and 3.8% against 1.4% and 1.0%), because 76% of the stations there are isolated against 9% in Paris. The level depends on the radius, but the gap is still there at 300 m and 500 m, and without the event days.

- *What the events showed (section 6) :* The 2 closures are clearly visible at the right stations: the 110 listed stations were out of service 72% (E09) and 49% (E10) of the time, against 3.7% normally. On march and strike days, out-of-service stations are about 2 times more frequent than on the days around, and no source explains why.

- *What it means for a user ?* When a user finds a station empty or full, in about 7 cases out of 10 another station within 300 m can serve instead. In the other 3 cases there is none, mostly because the station is isolated, which is much more common outside Paris. We measure stations and not people, so this is a signal and not a count of people left without a bike.

**Limits.**
- We measure stations and not people: the distance is a straight line, and a failing station is not a person without a bike.
- The 300 m is a choice, and the result depends on it (tested at 200 m and 500 m in 7.4).
- With 1 capture every 5 minutes, failures shorter than about 10 minutes are invisible.
- Only 8 autumn weeks with unusual events, not a whole year.
- The "no alternative" analysis was done on VM1 only, not on VM2.
- A commune is not a catchment area, and the feed shows stations and not bikes.

**Lessons learned, by step of the data lifecycle:**
- **Collection.** Always-on cloud VMs, because a laptop sleeps and the feed keeps no history. A second VM was cheap insurance. A success code is not proof of good content.
- **Sampling.** We tested the 5-minute interval on our own archive instead of assuming it.
- **Cleaning.** "Flag, do not alter": 0 rows dropped. We checked claims with the data, including our own: the "92% lost" claim did not hold.
- **Storage.** The data on Zenodo (with a DOI) and the code on GitHub. The benchmark on the 25 million rows showed Parquet about 125 times smaller than CSV and more than 10 times faster than SQLite on scans, so we keep Parquet with DuckDB.
- **Analysis.** We checked our own numbers: the first normal level used 14 days instead of 29. Testing 200 m and 500 m showed that the result depends on the radius.
- **Scale.** 25 million rows do not fit in pandas: we query with DuckDB, aggregate once and save small tables.
- **Reproducibility.** A reproducible package let reviewers find a hidden problem in the data before it shaped our results.

## 10. The replay: the whole period in one minute  ·  for the presentation

Every station as a dot on a map of Paris and its surroundings (commune outlines, Paris emphasised, and the Seine), hour by hour from 20.09 to 16.11: **teal** = works, **magenta** = cannot rent (empty), **amber** = cannot take a return (full), **grey** = out of service (a station takes the state it was in for at least half of that hour). Under the map, the network-wide share failing, with a moving marker and the known events shaded. It is built from the small `station_hour` table, not the 25 M rows. The aim is to see the daily rhythm, the weekends, and the closures of the events, not to explain them: causes only as sourced context. The video is a **presentation asset**: the report does not depend on it.

**To make the video, change `RUN_REPLAY = False` to `RUN_REPLAY = True` by hand in the first code cell** (it is `False` by default, so *Run All* never renders it), then run the notebook again. The render uses one processor core: expect **4 to 25 minutes** on the full period depending on the laptop (22 min on a mid-range one; try `SAMPLE_DAYS` first). The first render also downloads the map background once (`data/external/map_background.json`, about 1 MB). The notebook does **not** play the video: open `outputs/replay_velib.mp4` from the file explorer (a `.gif` if `imageio-ffmpeg` is missing). Set `RUN_REPLAY` back to `False` afterwards.

In [ ]:
# Background of the map: commune outlines (Paris emphasised) and the Seine. Fetched once and cached in
# data/external/map_background.json (outlines: IGN via API Géo; river: OpenStreetMap contributors, ODbL).
# If the internet or a service is unavailable the replay still renders, with a plain background.
MAP_BG = EXT / "map_background.json"

HTTP_HEADERS = {"User-Agent": "velib-station-availability-student-project/1.0 (course project, ESSEC)", "Accept": "application/json"}

def _http_json(url, params=None, form=None, tries=4):
    for k in range(tries):
        try:
            r = requests.post(url, data=form, headers=HTTP_HEADERS, timeout=90) if form else requests.get(url, params=params, headers=HTTP_HEADERS, timeout=60)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            print(f"  background: retry {k + 1} for {url.split('/')[2]}: {str(e)[:70]}")
            time.sleep(min(20, 4 * (k + 1)))
    return None

def _outer_rings(geom):
    polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    return [[[round(x, 5), round(y, 5)] for x, y in p[0]] for p in polys]

def load_map_background():
    if MAP_BG.exists():
        return json.loads(MAP_BG.read_text())
    bg = {"communes": [], "paris": [], "seine": []}
    deps = sorted(communes["insee_code"].str[:2].unique()) if communes is not None else ["75", "92", "93", "94"]
    complete = True
    for dep in deps:
        js = _http_json(f"https://geo.api.gouv.fr/departements/{dep}/communes", {"fields": "code,contour", "format": "json"})
        if js is None:
            complete = False
            continue
        for c in js:
            if c.get("contour"):
                bg["paris" if c["code"] == "75056" else "communes"] += _outer_rings(c["contour"])
    q = '[out:json][timeout:60];way["waterway"="river"]["name"="La Seine"](48.70,2.00,49.05,2.70);out geom;'
    js = None
    for _mirror in ("https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"):
        js = _http_json(_mirror, form={"data": q}, tries=3)
        if js is not None:
            break
    if js is not None:
        bg["seine"] = [[[round(p["lon"], 5), round(p["lat"], 5)] for p in w["geometry"]] for w in js.get("elements", []) if w.get("geometry")]
    complete = complete and bool(bg["seine"])
    if complete:
        MAP_BG.write_text(json.dumps(bg))
        print(f"map background saved: {MAP_BG.relative_to(ROOT)} ({MAP_BG.stat().st_size / 1e6:.1f} MB)")
    else:
        print("map background incomplete (a service did not answer): this render uses what was found; run again later to complete and cache it.")
    return bg

PALETTE = {"ok": "#2a9d8f", "cannot rent (empty)": "#c2185b", "cannot take a return (full)": "#edae49", "out of service": "#6c757d"}

def build_replay():
    from matplotlib.colors import to_rgba
    from matplotlib.patches import Patch
    idx = sh[["date_utc", "hour_utc"]].drop_duplicates().sort_values(["date_utc", "hour_utc"]).reset_index(drop=True)
    idx["t"] = np.arange(len(idx))
    sidx = panel_stations[["station_id"]].reset_index(drop=True).assign(s=lambda d: np.arange(len(d)))
    d = sh.merge(idx, on=["date_utc", "hour_utc"]).merge(sidx, on="station_id")
    T, S = len(idx), len(sidx)
    A = {k: np.zeros((T, S), dtype="float32") for k in ["n", "n_empty", "n_full", "n_oos"]}
    for k in A:
        A[k][d["t"].to_numpy(), d["s"].to_numpy()] = d[k].to_numpy()
    n = A["n"]
    code_ = np.full((T, S), -1, dtype=np.int8)
    code_[n > 0] = 0
    code_[(n > 0) & (A["n_empty"] >= n / 2)] = 1
    code_[(n > 0) & (A["n_full"] >= n / 2)] = 2
    code_[(n > 0) & (A["n_oos"] >= n / 2)] = 3
    pal = np.array([(0, 0, 0, 0)] + [to_rgba(c) for c in PALETTE.values()])      # index = code + 1

    nu = shares(sh, ["date_utc", "hour_utc"]).merge(idx, on=["date_utc", "hour_utc"]).sort_values("t")
    nu["pct_oos"] = 100 * nu["n_oos"] / nu["n"]
    ts = (pd.to_datetime(idx["date_utc"]) + pd.to_timedelta(idx["hour_utc"], unit="h")).dt.tz_localize("UTC").dt.tz_convert("Europe/Paris")
    tsn = ts.dt.tz_localize(None)

    lat, lon = panel_stations["lat"].to_numpy(), panel_stations["lon"].to_numpy()
    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    gs = fig.add_gridspec(2, 2, height_ratios=[3.8, 1.0], width_ratios=[1.85, 1], hspace=0.35, wspace=0.03)
    ax_map, ax_info, ax_tl = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
    ax_map.set_aspect(1 / np.cos(np.radians(lat.mean())))
    ax_map.set_xticks([]); ax_map.set_yticks([])
    from matplotlib.collections import PolyCollection, LineCollection
    bg = load_map_background()
    if bg["communes"]:
        ax_map.add_collection(PolyCollection(bg["communes"], facecolors="#f1f3f5", edgecolors="#c3c9cf", linewidths=0.5, zorder=1))
    if bg["paris"]:
        ax_map.add_collection(PolyCollection(bg["paris"], facecolors="#e4e9ee", edgecolors="#55616d", linewidths=1.3, zorder=2))
    if bg["seine"]:
        ax_map.add_collection(LineCollection(bg["seine"], colors="#9cc3dd", linewidths=3, capstyle="round", zorder=3))
    sc = ax_map.scatter(lon, lat, s=9, c=[(0, 0, 0, 0)] * S, zorder=5)
    padx, pady = 0.02 * (lon.max() - lon.min()), 0.02 * (lat.max() - lat.min())
    ax_map.set_xlim(lon.min() - padx, lon.max() + padx); ax_map.set_ylim(lat.min() - pady, lat.max() + pady)
    ax_map.text(0.01, 0.01, "Outlines: IGN via API Géo. River: © OpenStreetMap contributors", transform=ax_map.transAxes, fontsize=6, color="#666", zorder=6)
    ax_info.axis("off")
    t_time = ax_info.text(0.0, 0.96, "", fontsize=16, fontweight="bold", va="top")
    t_counts = ax_info.text(0.0, 0.82, "", fontsize=12, va="top", family="monospace")
    t_events = ax_info.text(0.0, 0.50, "", fontsize=11, va="top", wrap=True)
    ax_info.legend(handles=[Patch(color=c, label=k) for k, c in PALETTE.items()], loc="lower left", fontsize=10, frameon=False)
    ax_tl.plot(tsn, nu["pct_cannot_rent"], color="#c2185b", lw=0.8, label="cannot rent")
    ax_tl.plot(tsn, nu["pct_cannot_return"], color="#edae49", lw=0.8, label="cannot take a return")
    ax_tl.plot(tsn, nu["pct_oos"], color="#6c757d", lw=0.8, label="out of service")
    shade_events(ax_tl, labels=True)
    ax_tl.set_ylabel("% of stations"); ax_tl.legend(loc="upper left", fontsize=7, ncol=3)
    vline = ax_tl.axvline(tsn.iloc[0], color="black", lw=1.2)

    def update(i):
        sc.set_facecolor(pal[code_[i] + 1])
        t_time.set_text(ts.iloc[i].strftime("%a %d %b %Y, %H:%M") + " (Paris)")
        c = code_[i]
        t_counts.set_text(f"works               {(c == 0).sum():5d}\ncannot rent         {(c == 1).sum():5d}\ncannot take return  {(c == 2).sum():5d}\nout of service      {(c == 3).sum():5d}\nno data             {(c == -1).sum():5d}")
        txt = ""
        if events is not None:
            day = pd.Timestamp(ts.iloc[i].date())
            act = events[(events["date_start"] <= day) & (events["date_end"] >= day)]
            txt = "\n".join(f"{r.event_id}: {r.label}" for r in act.itertuples())
        t_events.set_text(txt)
        vline.set_xdata([tsn.iloc[i], tsn.iloc[i]])
        return sc, vline

    return fig, update, ts

def make_replay(step_hours=1, fps=24):
    import matplotlib.animation as animation
    fig, update, ts = build_replay()
    frames = range(0, len(ts), step_hours)
    try:
        import imageio_ffmpeg
        plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
        writer, path = animation.FFMpegWriter(fps=fps, bitrate=3000), OUT / "replay_velib.mp4"
    except ImportError:
        print("imageio-ffmpeg not installed: writing a smaller GIF instead (pip install imageio-ffmpeg for mp4)")
        writer, path = animation.PillowWriter(fps=min(fps, 10)), OUT / "replay_velib.gif"
    anim = animation.FuncAnimation(fig, update, frames=frames, blit=False)
    t0 = time.time()
    anim.save(path, writer=writer, dpi=100, progress_callback=lambda i, n: print(f"  frame {i + 1}/{n}", end="\r") if i % 100 == 0 else None)
    plt.close(fig)
    print(f"\nwritten {path.relative_to(ROOT)}: {path.stat().st_size/1e6:.1f} MB, {time.time()-t0:.0f} s")

def save_snapshots(moments):
    # still frames for the report: moments = {"name": "2026-10-24 14:00" (Paris time), ...}
    fig, update, ts = build_replay()
    for name, m in moments.items():
        i = int((ts.dt.tz_localize(None) - pd.Timestamp(m)).abs().to_numpy().argmin())
        update(i)
        fig.savefig(OUT / f"fig_snapshot_{name}.png", dpi=150)
        print("saved", f"fig_snapshot_{name}.png", "at", ts.iloc[i])
    plt.close(fig)

if RUN_REPLAY:
    make_replay()
    # save_snapshots({"before_closure": "2026-10-22 18:00", "closure": "2026-10-24 14:00"})   # pick moments after looking at section 6
else:
    print("Replay switched off (RUN_REPLAY = False).")
